In [ ]:
# =====================================================================================
# HEASNet FINAL ALL-IN-ONE EXPERIMENT  (HyperKvasir + Kvasir-V2)
# One run -> every metric, table, graph (PNG 500 dpi), explainability, complexity, timing,
# statistics, near-duplicate audit, optional ablation / baselines / seeds / sweeps.
# Everything is printed in the notebook AND saved into one ZIP:
#       /kaggle/working/HEASNet_FINAL_ALL_RESULTS.zip
# Paste into ONE   (GPU on, both datasets attached) and run.
# Your original training code (split, model, optimiser, seed, schedule) is unchanged,
# so the main numbers reproduce  earlier run (small GPU non-determinism aside).
# =====================================================================================
import os, gc, sys, io, json, time, random, warnings, shutil, re, copy, zipfile, platform, datetime, math
from pathlib import Path
from collections import Counter
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from PIL import Image
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms as T
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support, classification_report,
    confusion_matrix, roc_curve, auc, roc_auc_score, cohen_kappa_score, matthews_corrcoef, log_loss,
    f1_score, precision_recall_curve, average_precision_score)
from scipy.stats import binomtest
warnings.filterwarnings("ignore")
T0 = time.time()


# ================================ SWITCHES (edit only here) ================================
QUICK = os.environ.get("HEAS_QUICK") == "1"        # tiny smoke test, ignore
SEED = 42
EPOCHS, HEAD_WARMUP_EPOCHS = 40, 5                  # 5 frozen-backbone epochs + 35 fine-tuning
IMG_SIZE, BATCH_SIZE, NUM_WORKERS = 224, 16, 2
LR_HEAD, LR_BACKBONE, WEIGHT_DECAY = 1e-4, 1e-5, 1e-5
LABEL_SMOOTHING, DROPOUT = 0.05, 0.30
EMBED_DIM, NUM_HEADS, WINDOW_SIZE = 512, 8, 4
VAL_FRAC = TEST_FRAC = 0.15
BOOTSTRAP_REPS = 1000
USE_PRETRAINED = True
DPI = 500                                           # every PNG is saved at 500 dpi
REUSE_CHECKPOINT = False                             # if best_HEASNet.pt already exists in the output folder, skip training
# --- explainability
GRADCAM_PER_CLASS = 2
FAITH_PER_CLASS, FAITH_STEPS = 5, 20                # deletion/insertion test
CORR_MAX_PER_CLASS = 150                            # images per class for Grad-CAM vs attention-branch correlation
# --- near-duplicate audit
DUP_HAMMING = 4
# --- heavy optional experiments (retrain extra models with the SAME protocol)
RUN_EXTRAS = True                                   # master switch for everything below
EXTRA_DATASETS = ["HyperKvasir"]                    # add "Kvasir-V2" if you have time
RUN_ABLATION = True                                 # 6 variants (Table 8)
RUN_BASELINES = True                                # ResNet50, EfficientNetV2-S, ConvNeXt-T, ViT-B/16 (Table 7, Table 11)
REPEAT_SEEDS = []                        # extra seeds for HEASNet (mean +- SD); [] to skip
RUN_SWEEPS = False                                  # learning-rate and optimiser sweeps (Tables 9, 10): many runs
SWEEP_EPOCHS = 15                                   # shorter schedule for the sweeps only
LR_SWEEP = [1e-3, 5e-4, 1e-4, 5e-5, 1e-5]
OPT_SWEEP = ["adam", "sgd", "rmsprop"]
TIME_BUDGET_HOURS = 10.5                            # extras stop cleanly before the Kaggle session limit
INCLUDE_ALL_CHECKPOINTS_IN_ZIP = False              # True = also zip baseline/ablation .pt files (big)

DATASETS = {
    "HyperKvasir": {"root": "/kaggle/input/datasets/visheshtanwar26/processed-datatset",
                    "layout": "predefined_train_test", "expected_classes": int(os.environ.get("HEAS_HK_N", "12"))},
    "Kvasir-V2":   {"root": "/kaggle/input/datasets/subhajeetdas/kvasir-v2-gastrointestinal-image-dataset/KVASIR V2",
                    "layout": "class_folders", "expected_classes": int(os.environ.get("HEAS_KV_N", "8"))},
}
OUTPUT_ROOT = "/kaggle/working/HEASNet_FINAL_ALL_RESULTS"
ZIP_BASE = "/kaggle/working/HEASNet_FINAL_ALL_RESULTS"
if QUICK:
    EPOCHS, HEAD_WARMUP_EPOCHS, BOOTSTRAP_REPS, NUM_WORKERS = 2, 1, 30, 0
    USE_PRETRAINED, SWEEP_EPOCHS, REPEAT_SEEDS = False, 1, [43]
    FAITH_PER_CLASS, CORR_MAX_PER_CLASS, RUN_SWEEPS, LR_SWEEP, OPT_SWEEP, DPI = 2, 10, True, [1e-3, 1e-4], ["sgd"], 80
    DATASETS["HyperKvasir"]["root"] = os.environ["HEAS_HK_ROOT"]; DATASETS["Kvasir-V2"]["root"] = os.environ["HEAS_KV_ROOT"]
    OUTPUT_ROOT = os.environ["HEAS_OUT"]; ZIP_BASE = OUTPUT_ROOT + "_zip"; EXTRA_DATASETS = ["HyperKvasir", "Kvasir-V2"]
os.makedirs(OUTPUT_ROOT, exist_ok=True)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
PIN_MEMORY = torch.cuda.is_available()

# ---------------------------------- tee: everything printed is also saved ----------------------------------
class Tee:
    def __init__(self, *s): self.s = s
    def write(self, x):
        for f in self.s:
            try: f.write(x)
            except Exception: pass
    def flush(self):
        for f in self.s:
            try: f.flush()
            except Exception: pass
_LOG = open(os.path.join(OUTPUT_ROOT, "FULL_NOTEBOOK_LOG.txt"), "w", encoding="utf-8")
sys.stdout = Tee(sys.__stdout__, _LOG)

def banner(t, ch="="):
    print("\n" + ch * 100 + f"\n{t}\n" + ch * 100)
def elapsed_h(): return (time.time() - T0) / 3600
def time_left_ok(need_hours=0.9): return (TIME_BUDGET_HOURS - elapsed_h()) > need_hours

banner("HEASNet FINAL ALL-IN-ONE EXPERIMENT")
print("Start:", datetime.datetime.now().isoformat(timespec="seconds"), "| Device:", DEVICE,
      "| GPU:", torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else "none")
print(f"torch {torch.__version__} | python {platform.python_version()} | epochs {EPOCHS} (head {HEAD_WARMUP_EPOCHS}) | batch {BATCH_SIZE} | seed {SEED} | PNG dpi {DPI}")
print("""
WHAT THIS RUN PRINTS AND SAVES (for BOTH datasets, every item also goes into the ZIP):
  A. dataset audit: totals, classes, per-class counts, split sizes, imbalance, corrupt files, overlap audit, manifest, class-distribution and sample-image figures
  B. training: per-epoch train/val loss + accuracy, best epoch, total/mean epoch time, GPU memory, learning curves, checkpoint
  C. metrics on TRAIN, VALIDATION and TEST: accuracy, macro P/R/F1, weighted F1, Cohen kappa, MCC, macro/micro ROC-AUC, log loss, Brier (probability MSE), ECE, MCE
  D. class-wise: precision, recall (+95% Wilson CI), specificity, F1, ROC-AUC, average precision, support; confusion matrices (counts + normalised); top confusions; misclassified gallery
  E. statistics: 1000x bootstrap 95% CI (accuracy, F1, AUC, kappa, MCC, ECE), binomial tests, exact McNemar vs majority baseline
  F. graphs (PNG 500 dpi): loss, accuracy, confusion, ROC (+zoom), precision-recall, class heat-map, reliability, risk-coverage, split comparison, CI forest plot
  G. explainable AI: Grad-CAM (correct + wrong), attention-branch maps (SE / window / axial), Grad-CAM-vs-branch correlation heat-map, deletion/insertion faithfulness score
  H. near-duplicate audit (dHash) inside each dataset and between Kvasir-V2 and HyperKvasir, accuracy on duplicate vs non-duplicate test images
  I. complexity and speed: parameters, model size, FLOPs/GMACs (total + per module), latency (batch 1/8/16/32, fp32 + AMP, CPU), throughput, peak GPU memory (inference + training)
  J. optional: ablation (6 variants), baselines (ResNet50, EfficientNetV2-S, ConvNeXt-T, ViT-B/16), extra seeds, LR/optimiser sweeps, McNemar between models
  K. cross-dataset comparison tables/figures, PAPER_READY_TABLES.txt, manuscript_numbers.json, ALL_RESULTS_TABLES.xlsx, FULL_NOTEBOOK_LOG.txt
""")

def seed_everything(seed=SEED):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = True

def savefig(fig, path, show=True):
    fig.savefig(path, dpi=DPI, bbox_inches="tight")
    if show: plt.show()
    plt.close(fig)

plt.rcParams.update({"font.size": 10, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.alpha": .25, "figure.dpi": 100})
C_TRAIN, C_VAL, C_TEST = "#1f6f8b", "#d1603d", "#2a9d58"

# ======================================== DATA ========================================
IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}
def normalize_class_name(n): return n.lower().strip().replace(" ", "-").replace("_", "-")
def is_aug_path(p): return "_aug_" in Path(p).stem

def collect_class_folder_images(split_root, split_name):
    recs = []
    for p in Path(split_root).rglob("*"):
        if p.is_file() and p.suffix.lower() in IMAGE_EXTS:
            rel = p.relative_to(split_root)
            if len(rel.parts) < 2: continue
            recs.append((str(p), normalize_class_name(rel.parts[0]), split_name))
    return recs

def discover_hyperkvasir(root):
    root = Path(root); tr, te = root / "train", root / "test"
    if not tr.exists() or not te.exists(): raise FileNotFoundError(f"Expected {tr} and {te}")
    df = pd.DataFrame(collect_class_folder_images(tr, "train") + collect_class_folder_images(te, "test"),
                      columns=["path", "label", "source_split"]).drop_duplicates("path")
    if df.empty: raise RuntimeError("No HyperKvasir images found.")
    print(f"HyperKvasir: {len(df):,} images | {df.label.nunique()} classes")
    print("Class counts by provided split:"); print(pd.crosstab(df.label, df.source_split).to_string())
    return df.reset_index(drop=True)

def discover_kvasir_v2(root):
    recs = []
    for p in Path(root).rglob("*"):
        if p.is_file() and p.suffix.lower() in IMAGE_EXTS:
            rel = p.relative_to(root)
            if len(rel.parts) < 2: continue
            recs.append((str(p), normalize_class_name(rel.parts[0])))
    df = pd.DataFrame(recs, columns=["path", "label"]).drop_duplicates("path")
    if df.empty: raise RuntimeError("No Kvasir-V2 images found.")
    print(f"Kvasir-V2: {len(df):,} images | {df.label.nunique()} classes")
    print("Per-class image counts:"); print(df.label.value_counts().sort_index().to_string())
    return df.reset_index(drop=True)

def create_kvasir_splits(df):
    df = df.copy(); classes = sorted(df.label.unique()); m = {c: i for i, c in enumerate(classes)}
    df["target"] = df.label.map(m).astype(int)
    tv, te = train_test_split(df, test_size=TEST_FRAC, stratify=df.target, random_state=SEED)
    tr, va = train_test_split(tv, test_size=VAL_FRAC / (1 - TEST_FRAC), stratify=tv.target, random_state=SEED)
    return classes, tr.reset_index(drop=True), va.reset_index(drop=True), te.reset_index(drop=True)

def create_hyperkvasir_splits(df):
    """Supplied test set preserved; train folder split into train/val with an original image and its *_aug_* copies kept together."""
    df = df.copy(); classes = sorted(df.label.unique()); m = {c: i for i, c in enumerate(classes)}
    df["target"] = df.label.map(m).astype(int)
    ot = df[df.source_split == "train"].copy(); test_df = df[df.source_split == "test"].copy()
    ot["group_id"] = ot["path"].map(lambda p: re.sub(r"_aug_.*$", "", Path(p).stem))
    best, best_score = None, float("inf")
    for sd in range(SEED, SEED + 100):
        tri, vai = next(GroupShuffleSplit(n_splits=1, test_size=VAL_FRAC, random_state=sd).split(ot, ot.target, groups=ot.group_id))
        ctr, cva = ot.iloc[tri].copy(), ot.iloc[vai].copy()
        if set(ctr.label) != set(classes) or set(cva.label) != set(classes): continue
        sc = abs(len(cva) / len(ot) - VAL_FRAC)
        if sc < best_score: best_score, best = sc, (ctr, cva)
    if best is None: raise RuntimeError("Could not create a group-aware validation split containing every class.")
    tr, va = best
    if set(tr.group_id) & set(va.group_id): raise RuntimeError("Augmentation-group leakage detected.")
    return (classes, tr.drop(columns=["group_id"]).reset_index(drop=True),
            va.drop(columns=["group_id"]).reset_index(drop=True), test_df.reset_index(drop=True))

def verify_images(df, name, out_dir):
    bad = []
    for r in df.itertuples(index=False):
        try:
            with Image.open(r.path) as im: im.verify()
        except Exception as e: bad.append({"path": r.path, "label": r.label, "error": repr(e)})
    pd.DataFrame(bad, columns=["path", "label", "error"]).to_csv(os.path.join(out_dir, "corrupt_or_unreadable_images.csv"), index=False)
    if bad:
        df = df[~df.path.isin({b["path"] for b in bad})].copy(); print(f"Removed {len(bad)} corrupt/unreadable files from {name}.")
    else: print("Image integrity check: all discovered files are readable.")
    return df.reset_index(drop=True), len(bad)

def build_split_audit(classes, tr, va, te, out_dir, ds_name):
    fr = {"train": tr, "validation": va, "test": te}
    ct = pd.DataFrame(index=classes)
    for k, f in fr.items():
        c = f.label.value_counts(); ct[k] = [int(c.get(x, 0)) for x in classes]
    ct.index.name = "class"; ct["total"] = ct.sum(axis=1)
    ct.to_csv(os.path.join(out_dir, "CLASS_COUNTS_EVERY_SPLIT.csv"))
    pct = ct[["train", "validation", "test"]].div(ct[["train", "validation", "test"]].sum(0), axis=1) * 100
    pct.to_csv(os.path.join(out_dir, "CLASS_PERCENTAGES_WITHIN_SPLIT.csv"))
    summ = pd.DataFrame([{"split": k, "images": len(f), "percentage_of_dataset": 100 * len(f) / (len(tr) + len(va) + len(te)),
        "number_of_classes_present": f.label.nunique(), "missing_classes": "; ".join([c for c in classes if c not in set(f.label)])} for k, f in fr.items()])
    summ.to_csv(os.path.join(out_dir, "SPLIT_SIZE_AND_CLASS_AUDIT.csv"), index=False)
    ps = {k: set(v.path) for k, v in fr.items()}
    ov = {"train_validation": len(ps["train"] & ps["validation"]), "train_test": len(ps["train"] & ps["test"]), "validation_test": len(ps["validation"] & ps["test"])}
    aud = {"class_names": classes, "number_of_classes": len(classes), "split_sizes": {k: len(v) for k, v in fr.items()}, "overlap_counts": ov,
           "all_classes_in_train": all(c in set(tr.label) for c in classes), "all_classes_in_validation": all(c in set(va.label) for c in classes),
           "all_classes_in_test": all(c in set(te.label) for c in classes)}
    json.dump(aud, open(os.path.join(out_dir, "SPLIT_INTEGRITY_AUDIT.json"), "w"), indent=2)
    print("\nCOMPLETE CLASS COUNTS"); print(ct.to_string()); print("\nSPLIT SIZE / CLASS COVERAGE"); print(summ.to_string(index=False))
    print("\nSPLIT OVERLAP AUDIT:", ov)
    if any(ov.values()): raise RuntimeError("Split leakage detected.")
    # class imbalance analysis
    imb = []
    for k in ["train", "validation", "test", "total"]:
        v = ct[k].values.astype(float)
        imb.append({"subset": k, "min_class": classes[int(v.argmin())], "min_count": int(v.min()), "max_class": classes[int(v.argmax())],
                    "max_count": int(v.max()), "imbalance_ratio_max_over_min": float(v.max() / max(v.min(), 1)),
                    "coefficient_of_variation": float(v.std() / v.mean())})
    imb = pd.DataFrame(imb); imb.to_csv(os.path.join(out_dir, "CLASS_IMBALANCE_ANALYSIS.csv"), index=False)
    print("\nCLASS IMBALANCE ANALYSIS"); print(imb.round(3).to_string(index=False))
    # figure: class distribution
    fig, ax = plt.subplots(figsize=(max(9, len(classes) * 0.8), 5.2)); bot = np.zeros(len(classes))
    for k, col in zip(["train", "validation", "test"], [C_TRAIN, C_VAL, C_TEST]):
        ax.bar(range(len(classes)), ct[k].values, bottom=bot, label=k, color=col); bot += ct[k].values
    ax.set_xticks(range(len(classes))); ax.set_xticklabels(classes, rotation=55, ha="right"); ax.set_ylabel("Images")
    ax.set_title(f"{ds_name}: class distribution per split"); ax.legend(); fig.tight_layout()
    savefig(fig, os.path.join(out_dir, "Fig_01_class_distribution.png"))
    return ct, summ, aud, imb

def sample_grid(tr, classes, out_dir, ds_name):
    n = len(classes); cols = 4; rows = int(math.ceil(n / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 3.2, rows * 3.0), squeeze=False)
    for a in axes.ravel(): a.axis("off")
    for i, c in enumerate(classes):
        sub = tr[(tr.label == c)]; sub = sub[~sub.path.map(is_aug_path)] if (~sub.path.map(is_aug_path)).any() else sub
        p = sub.path.iloc[0]; a = axes[i // cols, i % cols]
        a.imshow(Image.open(p).convert("RGB").resize((IMG_SIZE, IMG_SIZE))); a.set_title(c, fontsize=9)
    fig.suptitle(f"{ds_name}: one example per class (training set)"); fig.tight_layout()
    savefig(fig, os.path.join(out_dir, "Fig_02_sample_images.png"))

train_transform = T.Compose([T.Resize((IMG_SIZE, IMG_SIZE)), T.ToTensor(), T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])])
eval_transform = train_transform
class ImageDataset(Dataset):
    def __init__(self, frame, transform): self.paths = frame.path.tolist(); self.targets = frame.target.astype(int).tolist(); self.transform = transform
    def __len__(self): return len(self.paths)
    def __getitem__(self, i):
        with Image.open(self.paths[i]) as im: x = im.convert("RGB")
        return self.transform(x), self.targets[i]

def make_loaders(tr, va, te, seed=SEED):
    ds = [ImageDataset(f, train_transform if k == 0 else eval_transform) for k, f in enumerate([tr, va, te])]
    g = torch.Generator().manual_seed(seed)
    common = dict(batch_size=BATCH_SIZE, num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY, persistent_workers=(NUM_WORKERS > 0))
    return (DataLoader(ds[0], shuffle=True, generator=g, **common), DataLoader(ds[1], shuffle=False, **common),
            DataLoader(ds[2], shuffle=False, **common), ds)

# ======================================== MODELS ========================================
class SqueezeExcitation(nn.Module):
    def __init__(self, c, reduction=16):
        super().__init__(); h = max(c // reduction, 8); self.fc1 = nn.Linear(c, h); self.fc2 = nn.Linear(h, c)
    def forward(self, x):
        z = torch.sigmoid(self.fc2(F.relu(self.fc1(x.mean(dim=(2, 3))))))
        return x * z[:, :, None, None]

class WindowAttention(nn.Module):
    def __init__(self, dim, heads=8, window=4):
        super().__init__(); self.window = window
        self.attn = nn.MultiheadAttention(dim, heads, batch_first=True); self.norm = nn.LayerNorm(dim)
    def forward(self, x):
        b, c, h, w_ = x.shape; w = self.window
        ph, pw = (-h) % w, (-w_) % w
        if ph or pw: x = F.pad(x, (0, pw, 0, ph))
        hp, wp = h + ph, w_ + pw
        t = x.permute(0, 2, 3, 1).reshape(b, hp // w, w, wp // w, w, c).permute(0, 1, 3, 2, 4, 5).reshape(-1, w * w, c)
        t = self.norm(t + self.attn(t, t, t, need_weights=False)[0])
        x = t.reshape(b, hp // w, wp // w, w, w, c).permute(0, 1, 3, 2, 4, 5).reshape(b, hp, wp, c).permute(0, 3, 1, 2)
        return x[:, :, :h, :w_]

class AxialAttentionHeight(nn.Module):
    def __init__(self, dim, heads=8):
        super().__init__(); self.attn = nn.MultiheadAttention(dim, heads, batch_first=True); self.norm = nn.LayerNorm(dim)
    def forward(self, x):
        b, c, h, w = x.shape
        t = x.permute(0, 3, 2, 1).reshape(b * w, h, c)
        t = self.norm(t + self.attn(t, t, t, need_weights=False)[0])
        return t.reshape(b, w, h, c).permute(0, 3, 2, 1)

class HEASNet(nn.Module):
    """EfficientNetV2-S + 1x1 projection + SE + window attention + height-axis attention + GAP + dropout + linear.
    use_* flags only switch modules off for the ablation study (all True = proposed model)."""
    def __init__(self, num_classes, pretrained=True, use_se=True, use_window=True, use_axial=True):
        super().__init__()
        w = models.EfficientNet_V2_S_Weights.DEFAULT if pretrained else None
        self.backbone = models.efficientnet_v2_s(weights=w).features
        self.projection = nn.Conv2d(1280, EMBED_DIM, 1, bias=False)
        self.se = SqueezeExcitation(EMBED_DIM) if use_se else nn.Identity()
        self.window_attention = WindowAttention(EMBED_DIM, NUM_HEADS, WINDOW_SIZE) if use_window else nn.Identity()
        self.axial_attention = AxialAttentionHeight(EMBED_DIM, NUM_HEADS) if use_axial else nn.Identity()
        self.dropout = nn.Dropout(DROPOUT); self.classifier = nn.Linear(EMBED_DIM, num_classes)
    def forward(self, x):
        x = self.projection(self.backbone(x)); x = self.se(x); x = self.window_attention(x); x = self.axial_attention(x)
        return self.classifier(self.dropout(x.mean(dim=(2, 3))))
    def backbone_parameters(self): return list(self.backbone.parameters())
    def head_parameters(self):
        return [p for m in [self.projection, self.se, self.window_attention, self.axial_attention, self.classifier] for p in m.parameters()]

class BaselineNet(nn.Module):
    """Standard torchvision classifiers for the retrained baselines (same protocol: frozen body, then fine-tuning)."""
    def __init__(self, name, num_classes, pretrained=True):
        super().__init__(); self.name = name
        W = lambda e: (e.DEFAULT if pretrained else None)
        if name == "ResNet50":
            self.net = models.resnet50(weights=W(models.ResNet50_Weights)); self.net.fc = nn.Linear(self.net.fc.in_features, num_classes); self.head = self.net.fc
        elif name == "EfficientNetV2-S":
            self.net = models.efficientnet_v2_s(weights=W(models.EfficientNet_V2_S_Weights))
            self.net.classifier[-1] = nn.Linear(self.net.classifier[-1].in_features, num_classes); self.head = self.net.classifier
        elif name == "ConvNeXt-T":
            self.net = models.convnext_tiny(weights=W(models.ConvNeXt_Tiny_Weights))
            self.net.classifier[-1] = nn.Linear(self.net.classifier[-1].in_features, num_classes); self.head = self.net.classifier
        elif name == "ViT-B/16":
            self.net = models.vit_b_16(weights=W(models.ViT_B_16_Weights))
            self.net.heads.head = nn.Linear(self.net.heads.head.in_features, num_classes); self.head = self.net.heads
        else: raise ValueError(name)
    def forward(self, x): return self.net(x)
    def head_parameters(self): return list(self.head.parameters())
    def backbone_parameters(self):
        hid = {id(p) for p in self.head.parameters()}; return [p for p in self.net.parameters() if id(p) not in hid]

# ======================================== TRAINING ========================================
def make_optimizer(name, groups, wd=WEIGHT_DECAY):
    name = name.lower()
    if name == "adamw": return torch.optim.AdamW(groups, weight_decay=wd)
    if name == "adam": return torch.optim.Adam(groups, weight_decay=wd)
    if name == "sgd": return torch.optim.SGD(groups, momentum=0.9, weight_decay=wd)
    if name == "rmsprop": return torch.optim.RMSprop(groups, weight_decay=wd)
    raise ValueError(name)

def run_train_epoch(model, loader, crit, opt, scaler):
    model.train(); tl = tc = tn = 0
    for x, y in loader:
        x = x.to(DEVICE, non_blocking=True); y = y.to(DEVICE, non_blocking=True)
        opt.zero_grad(set_to_none=True)
        with torch.autocast(device_type=DEVICE.type, enabled=(DEVICE.type == "cuda")):
            lo = model(x); loss = crit(lo, y)
        scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
        tl += loss.item() * y.size(0); tc += (lo.argmax(1) == y).sum().item(); tn += y.size(0)
    return tl / tn, tc / tn

@torch.no_grad()
def evaluate_epoch(model, loader, crit):
    model.eval(); tl = tc = tn = 0
    for x, y in loader:
        x = x.to(DEVICE, non_blocking=True); y = y.to(DEVICE, non_blocking=True)
        with torch.autocast(device_type=DEVICE.type, enabled=(DEVICE.type == "cuda")):
            lo = model(x); loss = crit(lo, y)
        tl += loss.item() * y.size(0); tc += (lo.argmax(1) == y).sum().item(); tn += y.size(0)
    return tl / tn, tc / tn

def train_model(model, tr_loader, va_loader, out_dir, tag="HEASNet", lr_head=LR_HEAD, lr_bb=LR_BACKBONE, opt_name="adamw",
                epochs=EPOCHS, warmup=HEAD_WARMUP_EPOCHS, verbose=True, history_name="training_history.csv", ckpt_name=None):
    model.to(DEVICE); crit = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)
    scaler = torch.amp.GradScaler("cuda", enabled=(DEVICE.type == "cuda"))
    best_path = os.path.join(out_dir, ckpt_name or f"best_{tag}.pt"); hist = []; best_acc, best_ep = -1.0, 0
    for p in model.backbone_parameters(): p.requires_grad = False
    head_opt = make_optimizer(opt_name, [{"params": model.head_parameters(), "lr": lr_head}])
    fine_opt = make_optimizer(opt_name, [{"params": model.backbone_parameters(), "lr": lr_bb}, {"params": model.head_parameters(), "lr": lr_head}])
    if DEVICE.type == "cuda": torch.cuda.reset_peak_memory_stats()
    for ep in range(1, epochs + 1):
        if ep == warmup + 1:
            for p in model.backbone_parameters(): p.requires_grad = True
        opt = head_opt if ep <= warmup else fine_opt; t0 = time.time()
        trl, tra = run_train_epoch(model, tr_loader, crit, opt, scaler); val, vaa = evaluate_epoch(model, va_loader, crit)
        if DEVICE.type == "cuda": torch.cuda.synchronize()
        sec = time.time() - t0
        hist.append({"epoch": ep, "epoch_seconds": sec, "train_loss": trl, "train_accuracy": tra, "val_loss": val, "val_accuracy": vaa,
                     "learning_rate_head": lr_head, "learning_rate_backbone": 0.0 if ep <= warmup else lr_bb})
        if vaa > best_acc: best_acc, best_ep = vaa, ep; torch.save(model.state_dict(), best_path)
        if verbose or ep == epochs:
            print(f"[{tag}] Epoch {ep:02d}/{epochs} | Train loss {trl:.4f} acc {tra:.4f} | Val loss {val:.4f} acc {vaa:.4f} | Best val {best_acc:.4f} (epoch {best_ep}) | {sec:.1f}s")
    model.load_state_dict(torch.load(best_path, map_location=DEVICE, weights_only=True))
    h = pd.DataFrame(hist); h.to_csv(os.path.join(out_dir, history_name), index=False)
    peak = float(torch.cuda.max_memory_allocated() / 1e6) if DEVICE.type == "cuda" else None
    return model, h, best_ep, best_acc, best_path, peak

# ======================================== PREDICTION AND METRICS ========================================
@torch.no_grad()
def predict(model, loader):
    model.eval(); P, Y = [], []
    for x, y in loader:
        x = x.to(DEVICE, non_blocking=True)
        with torch.autocast(device_type=DEVICE.type, enabled=(DEVICE.type == "cuda")): lo = model(x)
        P.append(torch.softmax(lo.float(), 1).cpu().numpy()); Y.append(y.numpy())
    return np.concatenate(P), np.concatenate(Y)

def ece_mce(probs, y, bins=15):
    conf = probs.max(1); ok = (probs.argmax(1) == y).astype(float); e = np.linspace(0, 1, bins + 1); ece = mce = 0.0
    for lo, hi in zip(e[:-1], e[1:]):
        m = (conf > lo) & (conf <= hi)
        if m.any():
            gap = abs(ok[m].mean() - conf[m].mean()); ece += m.mean() * gap; mce = max(mce, gap)
    return float(ece), float(mce)

def wilson_interval(k, n, z=1.96):
    if n == 0: return np.nan, np.nan
    p = k / n; d = 1 + z * z / n; c = (p + z * z / (2 * n)) / d; h = z * np.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d
    return float(c - h), float(c + h)

def compute_metrics(y, probs, classes):
    K = len(classes); pred = probs.argmax(1)
    pr, rc, f1, sup = precision_recall_fscore_support(y, pred, labels=np.arange(K), zero_division=0)
    cm = confusion_matrix(y, pred, labels=np.arange(K)); spec = []
    for k in range(K):
        tp = cm[k, k]; fn = cm[k].sum() - tp; fp = cm[:, k].sum() - tp; tn = cm.sum() - tp - fn - fp
        spec.append(tn / (tn + fp) if (tn + fp) else np.nan)
    yb = np.eye(K)[y]; aucs, aps = [], []
    for k in range(K):
        try: aucs.append(roc_auc_score(yb[:, k], probs[:, k]))
        except ValueError: aucs.append(np.nan)
        try: aps.append(average_precision_score(yb[:, k], probs[:, k]))
        except ValueError: aps.append(np.nan)
    macro_auc = float(np.nanmean(aucs)) if np.isfinite(aucs).any() else np.nan
    try: micro_auc = float(roc_auc_score(yb.ravel(), probs.ravel()))
    except ValueError: micro_auc = np.nan
    ece, mce = ece_mce(probs, y)
    m = {"n": int(len(y)), "accuracy": float(accuracy_score(y, pred)), "macro_precision": float(np.mean(pr)), "macro_recall": float(np.mean(rc)),
         "macro_f1": float(np.mean(f1)), "weighted_f1": float(f1_score(y, pred, average="weighted", zero_division=0)),
         "cohen_kappa": float(cohen_kappa_score(y, pred)), "mcc": float(matthews_corrcoef(y, pred)), "macro_roc_auc_ovr": macro_auc,
         "micro_roc_auc": micro_auc, "log_loss": float(log_loss(y, probs, labels=np.arange(K))),
         "brier_score_probability_MSE": float(np.mean(np.sum((probs - yb) ** 2, 1))), "ece_15_bins": ece, "mce_15_bins": mce,
         "macro_average_precision": float(np.nanmean(aps))}
    lo_hi = [wilson_interval(int(round(r * s)), int(s)) for r, s in zip(rc, sup)]
    pcl = pd.DataFrame({"class": classes, "precision": pr, "recall_sensitivity": rc, "recall_CI95_low": [a for a, _ in lo_hi],
        "recall_CI95_high": [b for _, b in lo_hi], "specificity": spec, "f1_score": f1, "roc_auc_ovr": aucs, "average_precision": aps, "support": sup})
    return m, pcl, cm

def bootstrap_all(y, probs, reps=BOOTSTRAP_REPS, seed=SEED):
    rng = np.random.default_rng(seed); n = len(y); K = probs.shape[1]; store = {k: [] for k in ["accuracy", "macro_f1", "macro_roc_auc_ovr", "cohen_kappa", "mcc", "ece_15_bins"]}
    for _ in range(reps):
        i = rng.integers(0, n, n); yy, pp = y[i], probs[i]; pr = pp.argmax(1)
        try:
            store["accuracy"].append(accuracy_score(yy, pr)); store["macro_f1"].append(f1_score(yy, pr, average="macro", zero_division=0))
            store["cohen_kappa"].append(cohen_kappa_score(yy, pr)); store["mcc"].append(matthews_corrcoef(yy, pr)); store["ece_15_bins"].append(ece_mce(pp, yy)[0])
            store["macro_roc_auc_ovr"].append(roc_auc_score(yy, pp, multi_class="ovr", average="macro", labels=np.arange(K)))
        except Exception: pass
    return {k: [float(np.percentile(v, 2.5)), float(np.percentile(v, 97.5))] if v else [np.nan, np.nan] for k, v in store.items()}

def mcnemar_exact(correct_a, correct_b):
    a = int(np.sum(correct_a & ~correct_b)); b = int(np.sum(~correct_a & correct_b)); d = a + b
    return {"only_A_correct": a, "only_B_correct": b, "discordant": d, "exact_p": float(binomtest(a, d, 0.5).pvalue) if d else 1.0}

# ======================================== PLOTS ========================================
def plot_training(h, out_dir, ds, best_ep, warmup=HEAD_WARMUP_EPOCHS):
    ep = h.epoch
    for key, ylab, fn, ttl in [("loss", "Cross-entropy loss", "Fig_03_loss_curve.png", "Training and validation loss"),
                               ("accuracy", "Accuracy", "Fig_04_accuracy_curve.png", "Training and validation accuracy")]:
        fig, ax = plt.subplots(figsize=(7, 4.5))
        ax.plot(ep, h[f"train_{key}"], color=C_TRAIN, lw=2, label="Training"); ax.plot(ep, h[f"val_{key}"], color=C_VAL, lw=2, label="Validation")
        ax.axvline(warmup + .5, color="gray", ls=":"); ax.text(warmup + .8, ax.get_ylim()[0] + (ax.get_ylim()[1] - ax.get_ylim()[0]) * .03, "fine-tuning starts", fontsize=8, color="gray")
        if key == "accuracy": ax.scatter([best_ep], [h.val_accuracy[best_ep - 1]], color=C_VAL, zorder=5); ax.annotate(f"selected epoch {best_ep}", (best_ep, h.val_accuracy[best_ep - 1]), xytext=(-80, -25), textcoords="offset points", fontsize=8, color=C_VAL, arrowprops=dict(arrowstyle="-", color=C_VAL))
        ax.set(xlabel="Epoch", ylabel=ylab, title=f"{ds}: {ttl}"); ax.legend(); fig.tight_layout(); savefig(fig, os.path.join(out_dir, fn))
    fig, axs = plt.subplots(2, 2, figsize=(11, 7.5))
    axs[0, 0].plot(ep, h.train_loss, color=C_TRAIN, label="train"); axs[0, 0].plot(ep, h.val_loss, color=C_VAL, label="val"); axs[0, 0].set_title("Loss"); axs[0, 0].legend()
    axs[0, 1].plot(ep, h.train_accuracy, color=C_TRAIN, label="train"); axs[0, 1].plot(ep, h.val_accuracy, color=C_VAL, label="val"); axs[0, 1].set_title("Accuracy"); axs[0, 1].legend()
    axs[1, 0].plot(ep, h.train_accuracy - h.val_accuracy, color="#7b2d8e"); axs[1, 0].axhline(0, color="k", lw=.5); axs[1, 0].set_title("Generalisation gap (train acc - val acc)")
    axs[1, 1].bar(ep, h.epoch_seconds, color="#888"); axs[1, 1].set_title("Seconds per epoch")
    for a in axs.ravel(): a.set_xlabel("Epoch")
    fig.suptitle(f"{ds}: learning dynamics"); fig.tight_layout(); savefig(fig, os.path.join(out_dir, "Fig_05_learning_dynamics.png"))

def plot_confusion(cm, classes, out_dir, normalized, split="test", num="06"):
    M = cm.astype(float)
    if normalized: rs = M.sum(1, keepdims=True); M = np.divide(M, rs, out=np.zeros_like(M), where=rs != 0)
    n = len(classes); fig, ax = plt.subplots(figsize=(max(8, n * .8), max(7, n * .72)))
    im = ax.imshow(M, cmap="Blues"); ax.grid(False)
    ax.set_xticks(range(n)); ax.set_yticks(range(n)); ax.set_xticklabels(classes, rotation=55, ha="right", fontsize=8); ax.set_yticklabels(classes, fontsize=8)
    th = np.nanmax(M) / 2
    for i in range(n):
        for j in range(n):
            if normalized and M[i, j] == 0: continue
            ax.text(j, i, f"{M[i,j]:.2f}" if normalized else f"{int(cm[i,j])}", ha="center", va="center", color="white" if M[i, j] > th else "black", fontsize=7)
    ax.set_xlabel("Predicted class"); ax.set_ylabel("True class"); ax.set_title(f"{split.title()} confusion matrix ({'row-normalised' if normalized else 'counts'})")
    fig.colorbar(im, fraction=.046); fig.tight_layout()
    savefig(fig, os.path.join(out_dir, f"Fig_{num}_confusion_{'normalized' if normalized else 'counts'}_{split}.png"))

def plot_roc_pr(y, probs, classes, out_dir):
    K = len(classes); yb = np.eye(K)[y]; rows = []
    fig, axs = plt.subplots(1, 2, figsize=(15, 6.5)); cmap = plt.get_cmap("tab20")
    for k, nm in enumerate(classes):
        if len(np.unique(yb[:, k])) < 2: continue
        fpr, tpr, _ = roc_curve(yb[:, k], probs[:, k]); a = auc(fpr, tpr); rows.append({"class": nm, "auc_ovr": a})
        for ax in axs: ax.plot(fpr, tpr, lw=1.5, color=cmap(k), label=f"{nm} ({a:.3f})")
    fpr, tpr, _ = roc_curve(yb.ravel(), probs.ravel()); ma = auc(fpr, tpr)
    for ax in axs: ax.plot(fpr, tpr, "k--", lw=2, label=f"micro-average ({ma:.3f})"); ax.plot([0, 1], [0, 1], "k:", lw=1)
    axs[0].set(xlim=(0, 1), ylim=(0, 1.02), xlabel="False positive rate", ylabel="True positive rate", title="One-vs-rest ROC curves (test)")
    axs[1].set(xlim=(0, .2), ylim=(.8, 1.005), xlabel="False positive rate", ylabel="True positive rate", title="Zoom: top-left corner"); axs[0].legend(loc="lower right", fontsize=7)
    fig.tight_layout(); savefig(fig, os.path.join(out_dir, "Fig_09_ROC_curves_test.png")); pd.DataFrame(rows).to_csv(os.path.join(out_dir, "test_classwise_AUC.csv"), index=False)
    fig, ax = plt.subplots(figsize=(8, 6.5))
    for k, nm in enumerate(classes):
        if len(np.unique(yb[:, k])) < 2: continue
        p, r, _ = precision_recall_curve(yb[:, k], probs[:, k]); ax.plot(r, p, lw=1.5, color=cmap(k), label=f"{nm} (AP {average_precision_score(yb[:, k], probs[:, k]):.3f})")
    ax.set(xlabel="Recall", ylabel="Precision", xlim=(0, 1), ylim=(0, 1.02), title="One-vs-rest precision-recall curves (test)"); ax.legend(fontsize=7, loc="lower left")
    fig.tight_layout(); savefig(fig, os.path.join(out_dir, "Fig_10_precision_recall_curves_test.png"))

def plot_classwise(pcl, out_dir):
    cols = ["precision", "recall_sensitivity", "specificity", "f1_score", "roc_auc_ovr", "average_precision"]; d = pcl.set_index("class")[cols]
    fig, ax = plt.subplots(figsize=(8.5, max(5, len(d) * .5))); im = ax.imshow(d.values, cmap="YlGnBu", vmin=min(.6, np.nanmin(d.values)), vmax=1, aspect="auto"); ax.grid(False)
    ax.set_xticks(range(len(cols))); ax.set_xticklabels(["Precision", "Recall", "Specificity", "F1", "ROC-AUC", "Avg. precision"], rotation=30, ha="right"); ax.set_yticks(range(len(d))); ax.set_yticklabels(d.index)
    for i in range(d.shape[0]):
        for j in range(d.shape[1]): ax.text(j, i, f"{d.values[i,j]:.3f}", ha="center", va="center", fontsize=7, color="white" if d.values[i, j] > .93 else "black")
    ax.set_title("Class-wise metric heat-map (test)"); fig.colorbar(im, fraction=.04); fig.tight_layout(); savefig(fig, os.path.join(out_dir, "Fig_11_classwise_heatmap.png"))
    fig, ax = plt.subplots(figsize=(max(9, len(d) * .9), 5)); x = np.arange(len(pcl)); r = pcl.recall_sensitivity.values
    ax.bar(x - .2, pcl.f1_score, .4, color=C_TRAIN, label="F1"); ax.bar(x + .2, r, .4, color=C_VAL, label="Recall")
    ax.errorbar(x + .2, r, yerr=[np.maximum(0, r - pcl.recall_CI95_low.values), np.maximum(0, pcl.recall_CI95_high.values - r)], fmt="none", ecolor="k", capsize=3, lw=1)
    ax.set_xticks(x); ax.set_xticklabels(pcl["class"], rotation=50, ha="right"); ax.set_ylim(.5, 1.03); ax.set_title("Class-wise F1 and recall with 95% Wilson CI (test)"); ax.legend(); fig.tight_layout()
    savefig(fig, os.path.join(out_dir, "Fig_12_classwise_bars_recall_CI.png"))

def plot_calibration(y, probs, out_dir):
    conf = probs.max(1); ok = (probs.argmax(1) == y).astype(float); e = np.linspace(0, 1, 16); xs, ys, ns = [], [], []
    for lo, hi in zip(e[:-1], e[1:]):
        m = (conf > lo) & (conf <= hi)
        if m.any(): xs.append(conf[m].mean()); ys.append(ok[m].mean()); ns.append(int(m.sum()))
    fig, axs = plt.subplots(1, 3, figsize=(16, 4.8))
    axs[0].plot([0, 1], [0, 1], "k:", label="perfect"); axs[0].plot(xs, ys, "o-", color=C_TRAIN, label="HEASNet")
    for a, b, n in zip(xs, ys, ns): axs[0].annotate(str(n), (a, b), fontsize=6, xytext=(0, 5), textcoords="offset points", ha="center")
    axs[0].set(xlabel="Mean predicted confidence", ylabel="Observed accuracy", title="Reliability diagram (test; numbers = images per bin)", xlim=(0, 1), ylim=(0, 1.02)); axs[0].legend()
    b = np.linspace(min(conf.min(), .99), 1, 25); axs[1].hist(conf[ok == 1], bins=b, alpha=.7, color=C_TEST, label="correct"); axs[1].hist(conf[ok == 0], bins=b, alpha=.8, color=C_VAL, label="wrong")
    axs[1].set(xlabel="Confidence (max softmax)", ylabel="Images", title="Confidence of correct vs wrong predictions"); axs[1].legend()
    order = np.argsort(-conf); acc_c = np.cumsum(ok[order]) / np.arange(1, len(y) + 1); axs[2].plot(np.arange(1, len(y) + 1) / len(y), acc_c, color=C_TRAIN, lw=2)
    axs[2].set(xlabel="Coverage (fraction kept, most confident first)", ylabel="Accuracy on kept", title="Risk-coverage curve"); fig.tight_layout(); savefig(fig, os.path.join(out_dir, "Fig_13_calibration_reliability.png"))
    rows = []
    for t in [0, .5, .6, .7, .8, .9, .95]:
        m = conf >= t
        if m.sum(): rows.append({"confidence_threshold": t, "images_retained": int(m.sum()), "coverage_percent": 100 * m.mean(), "accuracy_on_retained_percent": 100 * ok[m].mean(), "errors_remaining": int((1 - ok[m]).sum())})
    sel = pd.DataFrame(rows); sel.to_csv(os.path.join(out_dir, "selective_accuracy_table.csv"), index=False)
    print("\nSELECTIVE ACCURACY (confidence threshold)"); print(sel.round(2).to_string(index=False))
    cal = {"mean_confidence": float(conf.mean()), "accuracy": float(ok.mean()), "mean_conf_correct": float(conf[ok == 1].mean()), "mean_conf_wrong": float(conf[ok == 0].mean()) if (ok == 0).any() else None}
    return sel, cal

def plot_split_comparison(split_metrics, out_dir, ds):
    keys = ["accuracy", "macro_f1", "macro_roc_auc_ovr", "cohen_kappa", "mcc"]; fig, ax = plt.subplots(figsize=(9.5, 5)); x = np.arange(len(keys))
    for j, (sp, col) in enumerate(zip(["train", "validation", "test"], [C_TRAIN, C_VAL, C_TEST])):
        v = [split_metrics[sp][k] for k in keys]; ax.bar(x + (j - 1) * .27, v, .27, color=col, label=sp)
        for a, b in zip(x + (j - 1) * .27, v): ax.text(a, b + .003, f"{b:.3f}", ha="center", fontsize=6, rotation=90)
    ax.set_xticks(x); ax.set_xticklabels(["Accuracy", "Macro F1", "Macro AUC", "Kappa", "MCC"]); ax.set_ylim(.8, 1.04); ax.set_title(f"{ds}: train vs validation vs test"); ax.legend(); fig.tight_layout()
    savefig(fig, os.path.join(out_dir, "Fig_15_train_val_test_comparison.png"))

def plot_ci_forest(ci, metrics, out_dir, ds):
    ks = list(ci.keys()); fig, ax = plt.subplots(figsize=(8, 3.8))
    for i, k in enumerate(ks):
        v = metrics[k]; lo, hi = ci[k]; ax.errorbar(v, i, xerr=[[v - lo], [hi - v]], fmt="o", color=C_TRAIN, capsize=4); ax.text(hi + .002, i, f"{v:.4f} [{lo:.4f}, {hi:.4f}]", va="center", fontsize=8)
    ax.set_yticks(range(len(ks))); ax.set_yticklabels(ks); ax.set_xlim(.8, 1.08); ax.set_title(f"{ds}: test metrics with 95% bootstrap CI"); ax.invert_yaxis(); fig.tight_layout(); savefig(fig, os.path.join(out_dir, "Fig_25_bootstrap_CI_forest.png"))

def top_confusions(cm, classes, out_dir, k=12):
    rows = [{"true_class": classes[i], "predicted_class": classes[j], "count": int(cm[i, j]), "percent_of_true_class": 100 * cm[i, j] / max(cm[i].sum(), 1)} for i in range(len(classes)) for j in range(len(classes)) if i != j and cm[i, j] > 0]
    d = pd.DataFrame(rows).sort_values("count", ascending=False).head(k); d.to_csv(os.path.join(out_dir, "top_confusions.csv"), index=False)
    print("\nTOP CONFUSIONS (test)"); print(d.round(2).to_string(index=False))
    if len(d):
        fig, ax = plt.subplots(figsize=(8, .45 * len(d) + 1.5)); lab = [f"{a} -> {b}" for a, b in zip(d.true_class, d.predicted_class)]
        ax.barh(range(len(d)), d["count"], color=C_VAL); ax.set_yticks(range(len(d))); ax.set_yticklabels(lab, fontsize=8); ax.invert_yaxis(); ax.set_xlabel("Images"); ax.set_title("Most frequent confusions (test)"); fig.tight_layout()
        savefig(fig, os.path.join(out_dir, "Fig_16_top_confusions.png"))
    return d

def load_rgb(p): return Image.open(p).convert("RGB").resize((IMG_SIZE, IMG_SIZE))

def misclassified_gallery(paths, y, probs, classes, out_dir, n=20):
    pred = probs.argmax(1); wrong = np.where(pred != y)[0]
    if len(wrong) == 0: return
    wrong = wrong[np.argsort(-probs[wrong].max(1))][:n]; cols = 5; rows = int(math.ceil(len(wrong) / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 3.2, rows * 3.4), squeeze=False)
    for a in axes.ravel(): a.axis("off")
    for k, i in enumerate(wrong):
        a = axes[k // cols, k % cols]; a.imshow(load_rgb(paths[i])); a.set_title(f"True: {classes[y[i]]}\nPred: {classes[pred[i]]} ({probs[i].max():.2f})", fontsize=7, color="red")
    fig.suptitle("Most confident misclassified test images"); fig.tight_layout(); savefig(fig, os.path.join(out_dir, "Fig_17_misclassified_gallery.png"))
_trapz = getattr(np, "trapezoid", None) or np.trapz

# ======================================== EXPLAINABLE AI ========================================
def _mm(c): return (c - c.amin((1, 2), keepdim=True)) / (c.amax((1, 2), keepdim=True) - c.amin((1, 2), keepdim=True) + 1e-8)
def _up(m): return F.interpolate(m, size=(IMG_SIZE, IMG_SIZE), mode="bilinear", align_corners=False)[:, 0]

class XAI:
    """Grad-CAM on the last EfficientNetV2-S feature block + spatial maps of the SE / window / axial branches."""
    def __init__(self, model):
        self.m = model; self.st = {}; self.h = []
        for p in model.parameters(): p.requires_grad_(True)
        self.h.append(model.backbone[-1].register_forward_hook(lambda mod, i, o: self.st.__setitem__("act", o)))
        self.names = []
        for n in ["se", "window_attention", "axial_attention"]:
            mod = getattr(model, n)
            if not isinstance(mod, nn.Identity):
                self.names.append(n); self.h.append(mod.register_forward_hook(lambda mod_, i, o, nn_=n: self.st.__setitem__(nn_, o.detach())))
    def close(self):
        for h in self.h: h.remove()
    def run(self, x, target=None):
        self.m.eval(); x = x.to(DEVICE)
        with torch.enable_grad():
            lo = self.m(x); pred = lo.argmax(1); tg = pred if target is None else torch.as_tensor(target, device=DEVICE)
            act = self.st["act"]; g = torch.autograd.grad(lo.gather(1, tg[:, None]).sum(), act)[0]
        cam = F.relu((g.mean((2, 3), keepdim=True) * act).sum(1, keepdim=True)).detach()
        cam = _mm(_up(cam)); maps = {n: _mm(_up(self.st[n].abs().mean(1, keepdim=True))) for n in self.names}
        return lo.detach(), cam.cpu().numpy(), {k: v.cpu().numpy() for k, v in maps.items()}

def _batch(ds, idx): return torch.stack([ds[i][0] for i in idx])

def gradcam_galleries(model, test_ds, probs, y, classes, out_dir):
    xai = XAI(model); pred = probs.argmax(1); K = len(classes)
    def collect(correct):
        sel = []
        for c in range(K):
            idx = np.where((y == c) & ((pred == c) if correct else (pred != c)))[0]
            if len(idx): sel.extend(idx[np.argsort(-probs[idx].max(1))][:GRADCAM_PER_CLASS].tolist())
        return sel
    for correct, fn, ttl in [(True, "Fig_18_gradcam_correct.png", "Grad-CAM, correctly classified test images"), (False, "Fig_19_gradcam_misclassified.png", "Grad-CAM, misclassified test images (explains the predicted class)")]:
        sel = collect(correct)
        if not sel: continue
        cols = 4; rows = int(math.ceil(len(sel) / cols)); fig, axes = plt.subplots(rows, cols, figsize=(cols * 3.6, rows * 3.5), squeeze=False)
        for a in axes.ravel(): a.axis("off")
        for s in range(0, len(sel), 16):
            ch = sel[s:s + 16]; lo, cam, _ = xai.run(_batch(test_ds, ch))
            for k, i in enumerate(ch):
                a = axes[(s + k) // cols, (s + k) % cols]; a.imshow(load_rgb(test_ds.paths[i])); a.imshow(cam[k], cmap="jet", alpha=.4)
                p = int(lo[k].argmax()); a.set_title(f"True: {classes[y[i]]}\nPred: {classes[p]}", fontsize=8, color="green" if p == y[i] else "red")
        fig.suptitle(ttl, fontsize=13); fig.tight_layout(); savefig(fig, os.path.join(out_dir, fn))
    xai.close()

def attention_analysis(model, test_ds, probs, y, classes, out_dir):
    """Grad-CAM vs spatial maps of SE / window / axial branches: overlay figure + per-class Pearson correlation heat-map."""
    xai = XAI(model); pred = probs.argmax(1); K = len(classes); rows = []; names = xai.names
    lab = {"se": "SE", "window_attention": "WIN", "axial_attention": "AX"}
    ex = []
    for c in range(K):
        idx = np.where(y == c)[0][:CORR_MAX_PER_CLASS]; vals = {n: [] for n in names}
        for s in range(0, len(idx), 16):
            ch = idx[s:s + 16]; _, cam, mp = xai.run(_batch(test_ds, ch), target=torch.as_tensor(y[ch]))
            cf = cam.reshape(len(ch), -1); cf = (cf - cf.mean(1, keepdims=True)) / (cf.std(1, keepdims=True) + 1e-8)
            for n in names:
                mf = mp[n].reshape(len(ch), -1); mf = (mf - mf.mean(1, keepdims=True)) / (mf.std(1, keepdims=True) + 1e-8); vals[n].extend((cf * mf).mean(1).tolist())
        r = {"class": classes[c], "n_images": len(idx)}
        for n in names: v = np.array(vals[n]); r[f"{lab[n]}_corr_mean"] = float(v.mean()); r[f"{lab[n]}_corr_sd"] = float(v.std())
        rows.append(r)
        ok = np.where((y == c) & (pred == c))[0]
        if len(ok) and len(ex) < 8: ex.append(int(ok[0]))
    d = pd.DataFrame(rows); d.to_csv(os.path.join(out_dir, "gradcam_vs_attention_branch_correlation.csv"), index=False)
    print("\nGRAD-CAM vs ATTENTION-BRANCH PEARSON CORRELATION (mean per class)"); print(d.round(3).to_string(index=False))
    cols = [f"{lab[n]}_corr_mean" for n in names]
    fig, ax = plt.subplots(figsize=(6.2, max(4.5, K * .45))); v = d[cols].values; lim = max(.3, np.nanmax(np.abs(v)))
    im = ax.imshow(v, cmap="RdBu_r", vmin=-lim, vmax=lim, aspect="auto"); ax.grid(False); ax.set_xticks(range(len(cols))); ax.set_xticklabels([lab[n] for n in names]); ax.set_yticks(range(K)); ax.set_yticklabels(classes)
    for i in range(K):
        for j in range(len(cols)): ax.text(j, i, f"{v[i,j]:.2f}", ha="center", va="center", fontsize=8)
    ax.set_title("Pearson correlation: Grad-CAM vs attention-branch map"); fig.colorbar(im, fraction=.05); fig.tight_layout(); savefig(fig, os.path.join(out_dir, "Fig_21_gradcam_attention_correlation_heatmap.png"))
    if ex:
        fig, axes = plt.subplots(len(ex), 2 + len(names), figsize=((2 + len(names)) * 2.8, len(ex) * 2.8), squeeze=False)
        for r_, i in enumerate(ex):
            _, cam, mp = xai.run(_batch(test_ds, [i]), target=torch.as_tensor([y[i]])); img = load_rgb(test_ds.paths[i])
            axes[r_, 0].imshow(img); axes[r_, 0].set_ylabel(classes[y[i]], fontsize=8); axes[r_, 1].imshow(img); axes[r_, 1].imshow(cam[0], cmap="jet", alpha=.45)
            for k, n in enumerate(names): axes[r_, 2 + k].imshow(img); axes[r_, 2 + k].imshow(mp[n][0], cmap="jet", alpha=.45)
            if r_ == 0:
                for k, t in enumerate(["Image", "Grad-CAM"] + [lab[n] + " branch" for n in names]): axes[0, k].set_title(t, fontsize=9)
        for a in axes.ravel(): a.set_xticks([]); a.set_yticks([]); a.grid(False)
        fig.suptitle("Grad-CAM and attention-branch maps (correct test images)"); fig.tight_layout(); savefig(fig, os.path.join(out_dir, "Fig_20_attention_branch_maps.png"))
    xai.close(); return d

@torch.no_grad()
def _prob_curve(model, x, order_rank, c, steps, mode):
    fr = np.linspace(0, 1, steps + 1); HW = IMG_SIZE * IMG_SIZE; imgs = []
    for f in fr:
        m = (order_rank < f * HW).reshape(1, 1, IMG_SIZE, IMG_SIZE).to(x.device)
        imgs.append(x * (~m) if mode == "deletion" else x * m)
    pr = torch.softmax(model(torch.cat(imgs)).float(), 1)[:, c].cpu().numpy(); return fr, pr

def faithfulness(model, test_ds, probs, y, classes, out_dir):
    """Deletion / insertion test of Grad-CAM (lower deletion AUC and higher insertion AUC = more faithful) against random maps."""
    xai = XAI(model); pred = probs.argmax(1); rows = []; curves = {k: [] for k in ["del_cam", "del_rand", "ins_cam", "ins_rand"]}
    g = torch.Generator().manual_seed(SEED)
    for c in range(len(classes)):
        idx = np.where((y == c) & (pred == c))[0][:FAITH_PER_CLASS]
        for i in idx:
            x = _batch(test_ds, [i]); _, cam, _ = xai.run(x); x = x.to(DEVICE); cam = torch.from_numpy(cam[0]).flatten()
            rk = lambda s: torch.argsort(torch.argsort(-s)).view(-1)
            r_cam = rk(cam); r_rand = rk(torch.rand(cam.numel(), generator=g)); out = {"class": classes[c]}
            for mode, tag in [("deletion", "del"), ("insertion", "ins")]:
                for nm, rr in [("cam", r_cam), ("rand", r_rand)]:
                    fr, pr = _prob_curve(model, x, rr, c, FAITH_STEPS, mode); out[f"{tag}_{nm}_auc"] = float(_trapz(pr, fr)); curves[f"{tag}_{nm}"].append(pr)
            rows.append(out)
    xai.close()
    if not rows: return None
    d = pd.DataFrame(rows); d.to_csv(os.path.join(out_dir, "faithfulness_deletion_insertion_per_image.csv"), index=False)
    s = d.groupby("class").mean(numeric_only=True); s.loc["ALL"] = d.mean(numeric_only=True); s.to_csv(os.path.join(out_dir, "faithfulness_deletion_insertion_summary.csv"))
    print("\nGRAD-CAM FAITHFULNESS (deletion AUC lower = better, insertion AUC higher = better; rand = random map)"); print(s.round(3).to_string())
    fr = np.linspace(0, 1, FAITH_STEPS + 1); fig, axs = plt.subplots(1, 2, figsize=(11, 4.4))
    for ax, tag, ttl in [(axs[0], "del", "Deletion: remove most-salient pixels first"), (axs[1], "ins", "Insertion: reveal most-salient pixels first")]:
        for nm, col, lb in [("cam", C_TRAIN, "Grad-CAM"), ("rand", "#888", "random map")]:
            a = np.array(curves[f"{tag}_{nm}"]); ax.plot(fr, a.mean(0), color=col, lw=2, label=f"{lb} (AUC {_trapz(a.mean(0), fr):.3f})"); ax.fill_between(fr, a.mean(0) - a.std(0), a.mean(0) + a.std(0), color=col, alpha=.15)
        ax.set(xlabel="Fraction of pixels", ylabel="Probability of predicted class", title=ttl); ax.legend()
    fig.tight_layout(); savefig(fig, os.path.join(out_dir, "Fig_22_gradcam_faithfulness_deletion_insertion.png")); return s

# ======================================== NEAR-DUPLICATES ========================================
HASH = {}
def dhash(path):
    if path in HASH: return HASH[path]
    with Image.open(path) as im: a = np.asarray(im.convert("L").resize((9, 8), Image.LANCZOS), dtype=np.int16)
    v = np.packbits((a[:, 1:] > a[:, :-1]).reshape(-1)).view(np.uint64)[0]; HASH[path] = v; return v

def hash_many(paths, label=""):
    out = np.empty(len(paths), dtype=np.uint64)
    for i, p in enumerate(paths):
        out[i] = dhash(p)
        if (i + 1) % 3000 == 0: print(f"   hashed {i+1}/{len(paths)} {label}")
    return out

def _popcount(x):
    x = x - ((x >> np.uint64(1)) & np.uint64(0x5555555555555555)); x = (x & np.uint64(0x3333333333333333)) + ((x >> np.uint64(2)) & np.uint64(0x3333333333333333))
    x = (x + (x >> np.uint64(4))) & np.uint64(0x0F0F0F0F0F0F0F0F); return (x * np.uint64(0x0101010101010101)) >> np.uint64(56)

def min_hamming(a, b, chunk=128):
    out = np.empty(len(a), dtype=np.int64)
    for s in range(0, len(a), chunk): out[s:s + chunk] = _popcount(a[s:s + chunk, None] ^ b[None, :]).min(1).astype(np.int64)
    return out

def near_duplicate_analysis(ds, tr, va, te, y_te, p_te, out_dir):
    banner(f"{ds}: NEAR-DUPLICATE AUDIT (64-bit dHash, Hamming <= {DUP_HAMMING})", "-")
    orig = tr[~tr.path.map(is_aug_path)]; ht = hash_many(orig.path.tolist(), "train originals"); hv = hash_many(va.path.tolist(), "validation"); he = hash_many(te.path.tolist(), "test")
    dte = min_hamming(he, ht); dva = min_hamming(hv, ht); ok = (p_te.argmax(1) == y_te)
    dup = dte <= DUP_HAMMING; ex = dte == 0
    res = {"train_original_images": int(len(orig)), "test_images": int(len(te)), "test_with_near_duplicate_in_train": int(dup.sum()), "test_near_duplicate_percent": float(100 * dup.mean()),
           "test_exact_duplicates": int(ex.sum()), "test_exact_duplicate_percent": float(100 * ex.mean()), "validation_with_near_duplicate_in_train": int((dva <= DUP_HAMMING).sum()),
           "validation_near_duplicate_percent": float(100 * (dva <= DUP_HAMMING).mean()),
           "accuracy_on_test_with_duplicate": float(ok[dup].mean()) if dup.any() else None, "accuracy_on_test_without_duplicate": float(ok[~dup].mean()) if (~dup).any() else None,
           "accuracy_all_test": float(ok.mean()), "n_test_without_duplicate": int((~dup).sum())}
    pd.DataFrame({"path": te.path, "min_hamming_to_train": dte, "near_duplicate": dup, "correct": ok}).to_csv(os.path.join(out_dir, "near_duplicate_per_test_image.csv"), index=False)
    json.dump(res, open(os.path.join(out_dir, "near_duplicate_summary.json"), "w"), indent=2); print(json.dumps(res, indent=2))
    fig, axs = plt.subplots(1, 2, figsize=(10, 4)); axs[0].hist(dte, bins=np.arange(0, 33), color=C_TRAIN); axs[0].axvline(DUP_HAMMING + .5, color="r", ls="--"); axs[0].set(xlabel="Min Hamming distance to a training image", ylabel="Test images", title="Near-duplicate distance")
    v = [100 * ok[dup].mean() if dup.any() else 0, 100 * ok[~dup].mean() if (~dup).any() else 0, 100 * ok.mean()]; axs[1].bar(["with duplicate", "without duplicate", "all test"], v, color=[C_VAL, C_TRAIN, C_TEST]); axs[1].set_ylim(80, 101); axs[1].set_ylabel("Accuracy (%)")
    for i, t in enumerate(v): axs[1].text(i, t + .2, f"{t:.2f}", ha="center")
    axs[1].set_title("Accuracy by duplicate status"); fig.tight_layout(); savefig(fig, os.path.join(out_dir, "Fig_23_near_duplicate_audit.png")); return res

# ======================================== COMPLEXITY AND SPEED ========================================
def compute_complexity(model, out_dir, tag="HEASNet", save=True):
    model.eval().to(DEVICE); x = torch.randn(1, 3, IMG_SIZE, IMG_SIZE, device=DEVICE)
    tot = sum(p.numel() for p in model.parameters()); res = {"model": tag, "parameters_total": int(tot), "parameters_trainable": int(sum(p.numel() for p in model.parameters() if p.requires_grad)),
        "parameters_million": tot / 1e6, "weights_fp32_MB": tot * 4 / 1e6, "weights_fp16_MB": tot * 2 / 1e6, "input_shape": [1, 3, IMG_SIZE, IMG_SIZE], "gmacs": None, "gflops": None, "flop_note": ""}
    try:
        from torch.utils.flop_counter import FlopCounterMode
        with torch.no_grad(), FlopCounterMode(display=False) as fc: model(x)
        fl = fc.get_total_flops(); res["gflops"] = fl / 1e9; res["gmacs"] = fl / 2e9
        res["flop_note"] = "torch.utils.flop_counter: counts matmul/conv/attention multiply-adds (1 MAC = 2 FLOPs); element-wise ops, norms and activations are not counted."
        if isinstance(model, HEASNet):
            br = []
            with torch.no_grad():
                feats = None
                def cnt(f, inp):
                    with FlopCounterMode(display=False) as c: o = f(inp)
                    return o, c.get_total_flops()
                h, f_ = cnt(model.backbone, x); br.append(("EfficientNetV2-S backbone", model.backbone, f_))
                h, f_ = cnt(model.projection, h); br.append(("1x1 projection", model.projection, f_))
                for nm, mod in [("SE", model.se), ("Window attention", model.window_attention), ("Axial attention (height)", model.axial_attention)]:
                    h, f_ = cnt(mod, h); br.append((nm, mod, f_))
                _, f_ = cnt(model.classifier, h.mean((2, 3))); br.append(("Classifier", model.classifier, f_))
            d = pd.DataFrame([{"module": n, "parameters": sum(p.numel() for p in m.parameters()), "GMACs": f / 2e9} for n, m, f in br]); d["percent_of_GMACs"] = 100 * d.GMACs / d.GMACs.sum()
            if save: d.to_csv(os.path.join(out_dir, f"complexity_breakdown_{tag}.csv"), index=False)
            print(f"\nCOMPLEXITY BREAKDOWN {tag}"); print(d.round(4).to_string(index=False)); res["breakdown"] = d.to_dict("records")
    except Exception as e:
        res["flop_note"] = f"FLOP counter failed: {e!r}"
        try:
            from fvcore.nn import FlopCountAnalysis
            fl = FlopCountAnalysis(model, x); res["gmacs"] = float(fl.total() / 1e9); res["gflops"] = 2 * res["gmacs"]; res["flop_note"] = "fvcore (MAC count)"
        except Exception as e2: res["flop_note"] += f" | fvcore unavailable: {e2!r}"
    if save: json.dump(res, open(os.path.join(out_dir, f"model_complexity_{tag}.json"), "w"), indent=2, default=str)
    print(f"{tag}: params {res['parameters_million']:.3f} M | fp32 {res['weights_fp32_MB']:.1f} MB | GMACs {res['gmacs']} | GFLOPs {res['gflops']}")
    return res

def _timeit(fn, repeats, warmup):
    for _ in range(warmup): fn()
    if DEVICE.type == "cuda": torch.cuda.synchronize()
    ts = []
    for _ in range(repeats):
        if DEVICE.type == "cuda": torch.cuda.synchronize()
        t0 = time.perf_counter(); fn()
        if DEVICE.type == "cuda": torch.cuda.synchronize()
        ts.append((time.perf_counter() - t0) * 1000)
    return np.array(ts)

def benchmark_speed(model, out_dir, tag="HEASNet", save=True):
    rep, wu = (60, 15) if not QUICK else (3, 1); rows = []; model.eval().to(DEVICE)
    for amp in ([False, True] if DEVICE.type == "cuda" else [False]):
        for bs in [1, 8, 16, 32]:
            x = torch.randn(bs, 3, IMG_SIZE, IMG_SIZE, device=DEVICE)
            if DEVICE.type == "cuda": torch.cuda.reset_peak_memory_stats()
            def f():
                with torch.inference_mode(), torch.autocast(device_type=DEVICE.type, enabled=amp and DEVICE.type == "cuda"): model(x)
            try: ts = _timeit(f, rep, wu)
            except RuntimeError as e: print("benchmark skipped", bs, e); continue
            rows.append({"model": tag, "device": str(DEVICE), "precision": "AMP fp16" if amp else "fp32", "batch_size": bs, "mean_ms_per_batch": ts.mean(), "median_ms_per_batch": np.median(ts),
                         "std_ms": ts.std(), "p95_ms": np.percentile(ts, 95), "mean_ms_per_image": ts.mean() / bs, "throughput_images_per_s": 1000 * bs / ts.mean(),
                         "peak_gpu_MB_inference": float(torch.cuda.max_memory_allocated() / 1e6) if DEVICE.type == "cuda" else None})
    try:
        cm_ = copy.deepcopy(model).cpu().float().eval(); x = torch.randn(1, 3, IMG_SIZE, IMG_SIZE)
        def fc_():
            with torch.inference_mode(): cm_(x)
        ts = _timeit(fc_, 3 if QUICK else 20, 1)
        rows.append({"model": tag, "device": f"CPU ({torch.get_num_threads()} threads)", "precision": "fp32", "batch_size": 1, "mean_ms_per_batch": ts.mean(), "median_ms_per_batch": np.median(ts), "std_ms": ts.std(),
                     "p95_ms": np.percentile(ts, 95), "mean_ms_per_image": ts.mean(), "throughput_images_per_s": 1000 / ts.mean(), "peak_gpu_MB_inference": None}); del cm_
    except Exception as e: print("CPU benchmark skipped:", e)
    d = pd.DataFrame(rows)
    if save: d.to_csv(os.path.join(out_dir, f"inference_benchmark_{tag}.csv"), index=False)
    print(f"\nINFERENCE SPEED {tag}"); print(d.round(3).to_string(index=False)); return d

def plot_efficiency(comp, bench, out_dir, ds):
    if "breakdown" not in comp: return
    d = pd.DataFrame(comp["breakdown"]); fig, axs = plt.subplots(1, 3, figsize=(16, 4.6))
    axs[0].barh(d.module, d.GMACs, color=C_TRAIN); axs[0].invert_yaxis(); axs[0].set_xlabel("GMACs"); axs[0].set_title("Compute per module (224x224)")
    axs[1].barh(d.module, d.parameters / 1e6, color=C_VAL); axs[1].invert_yaxis(); axs[1].set_xlabel("Parameters (M)"); axs[1].set_title("Parameters per module")
    g = bench[bench.device == str(DEVICE)] if len(bench) else bench
    for pr, col in [("fp32", C_TRAIN), ("AMP fp16", C_TEST)]:
        s = g[g.precision == pr]
        if len(s): axs[2].plot(s.batch_size, s.throughput_images_per_s, "o-", color=col, label=pr)
    axs[2].set(xlabel="Batch size", ylabel="Images per second", title="Throughput"); axs[2].legend(); fig.suptitle(f"{ds}: model complexity and speed"); fig.tight_layout()
    savefig(fig, os.path.join(out_dir, "Fig_24_complexity_and_speed.png"))

# ======================================== FULL EVALUATION OF ONE MODEL ========================================
def evaluate_and_report(model, loaders, dsets, classes, out_dir, ds, tr_df):
    banner(f"{ds}: FULL EVALUATION (train / validation / test)", "-")
    R = {}
    for sp, ld in zip(["train", "validation", "test"], loaders):
        P, Y = predict(model, ld); m, pcl, cm = compute_metrics(Y, P, classes); R[sp] = {"probs": P, "y": Y, "metrics": m, "pcl": pcl, "cm": cm}
        pcl.to_csv(os.path.join(out_dir, f"{sp}_classwise_metrics.csv"), index=False); pd.DataFrame(cm, index=classes, columns=classes).to_csv(os.path.join(out_dir, f"{sp}_confusion_matrix.csv"))
        np.savez_compressed(os.path.join(out_dir, f"predictions_{sp}.npz"), y_true=Y, probabilities=P, y_pred=P.argmax(1), class_names=np.asarray(classes), paths=np.asarray(dsets[["train", "validation", "test"].index(sp)].paths))
        print(f"\n{sp.upper()} METRICS"); print(pd.Series(m).round(5).to_string()); print(f"\n{sp.upper()} class-wise metrics:"); print(pcl.round(4).to_string(index=False))
    sm = pd.DataFrame([{"split": k, **R[k]["metrics"]} for k in ["train", "validation", "test"]]); sm.to_csv(os.path.join(out_dir, "all_split_metrics.csv"), index=False)
    gap = pd.DataFrame([{"metric": k, "train": R["train"]["metrics"][k], "validation": R["validation"]["metrics"][k], "test": R["test"]["metrics"][k], "train_minus_test": R["train"]["metrics"][k] - R["test"]["metrics"][k]} for k in ["accuracy", "macro_f1", "macro_roc_auc_ovr", "cohen_kappa", "mcc", "log_loss", "ece_15_bins"]])
    gap.to_csv(os.path.join(out_dir, "generalisation_gap_train_val_test.csv"), index=False); print("\nGENERALISATION GAP"); print(gap.round(4).to_string(index=False))
    plot_split_comparison({k: R[k]["metrics"] for k in ["train", "validation", "test"]}, out_dir, ds)
    y, P = R["test"]["y"], R["test"]["probs"]; pred = P.argmax(1); K = len(classes); tm = R["test"]["metrics"]
    # statistics
    ci = bootstrap_all(y, P); maj = Counter(y.tolist()).most_common(1)[0][1] / len(y); tmaj = int(tr_df.target.value_counts().idxmax())
    mc = mcnemar_exact(pred == y, np.full_like(y, tmaj) == y); acc_w = wilson_interval(int((pred == y).sum()), len(y))
    stat = {"bootstrap_reps": BOOTSTRAP_REPS, "bootstrap_95CI": ci, "accuracy_wilson_95CI": list(acc_w), "binomial_vs_uniform_chance_p": float(binomtest(int((pred == y).sum()), len(y), 1 / K, alternative="greater").pvalue),
            "binomial_vs_test_majority_rate_p": float(binomtest(int((pred == y).sum()), len(y), maj, alternative="greater").pvalue), "test_majority_class_rate": float(maj),
            "McNemar_vs_training_majority_baseline": {"baseline_class": classes[tmaj], "only_HEASNet_correct": mc["only_A_correct"], "only_baseline_correct": mc["only_B_correct"], "discordant": mc["discordant"], "exact_p": mc["exact_p"]}}
    json.dump(stat, open(os.path.join(out_dir, "statistical_tests.json"), "w"), indent=2); print("\nTEST STATISTICAL ANALYSIS"); print(json.dumps(stat, indent=2))
    pd.DataFrame([{"metric": k, "estimate": tm[k], "ci_low": v[0], "ci_high": v[1]} for k, v in ci.items()]).to_csv(os.path.join(out_dir, "bootstrap_confidence_intervals.csv"), index=False)
    pd.DataFrame(classification_report(y, pred, labels=np.arange(K), target_names=classes, output_dict=True, zero_division=0)).T.to_csv(os.path.join(out_dir, "test_classification_report_full.csv"))
    # figures
    plot_confusion(R["test"]["cm"], classes, out_dir, False); plot_confusion(R["test"]["cm"], classes, out_dir, True, num="07")
    plot_roc_pr(y, P, classes, out_dir); plot_classwise(R["test"]["pcl"], out_dir); sel, cal = plot_calibration(y, P, out_dir); plot_ci_forest(ci, tm, out_dir, ds)
    tc = top_confusions(R["test"]["cm"], classes, out_dir); misclassified_gallery(dsets[2].paths, y, P, classes, out_dir)
    R["stat"], R["ci"], R["cal"], R["top_conf"], R["selective"] = stat, ci, cal, tc, sel
    return R

# ======================================== ONE DATASET ========================================
CTX = {}
def run_dataset(ds, cfg):
    banner(f"STARTING DATASET: {ds}", "#"); seed_everything(SEED)
    od = os.path.join(OUTPUT_ROOT, ds.replace("-", "_")); os.makedirs(od, exist_ok=True)
    root = Path(cfg["root"])
    if not root.exists(): raise FileNotFoundError(f"Dataset path does not exist: {root}")
    print("Dataset found:", root)
    df = discover_hyperkvasir(root) if cfg["layout"] == "predefined_train_test" else discover_kvasir_v2(root)
    if df.label.nunique() != cfg["expected_classes"]: raise RuntimeError(f"{ds}: expected {cfg['expected_classes']} classes, found {df.label.nunique()}")
    df, n_bad = verify_images(df, ds, od)
    classes, tr, va, te = (create_hyperkvasir_splits if cfg["layout"] == "predefined_train_test" else create_kvasir_splits)(df)
    print(f"\nDATA SPLIT SUMMARY\nTotal: {len(df):,} | Train: {len(tr):,} ({len(tr)/len(df):.1%}) | Validation: {len(va):,} ({len(va)/len(df):.1%}) | Test: {len(te):,} ({len(te)/len(df):.1%})")
    if cfg["layout"] == "predefined_train_test": print(f"Augmented copies (*_aug_*): train {int(tr.path.map(is_aug_path).sum())} | validation {int(va.path.map(is_aug_path).sum())} | test {int(te.path.map(is_aug_path).sum())}")
    pd.concat([tr.assign(split="train"), va.assign(split="validation"), te.assign(split="test")]).to_csv(os.path.join(od, "exact_image_split_manifest.csv"), index=False)
    ct, ss, aud, imb = build_split_audit(classes, tr, va, te, od, ds); sample_grid(tr, classes, od, ds)
    trl, val, tel, dsets = make_loaders(tr, va, te)
    model = HEASNet(len(classes), USE_PRETRAINED); print(f"\nModel: HEASNet | classes {len(classes)} | parameters {sum(p.numel() for p in model.parameters())/1e6:.3f} M")
    print(f"Training protocol: {HEAD_WARMUP_EPOCHS} backbone-frozen epochs + {EPOCHS-HEAD_WARMUP_EPOCHS} fine-tuning epochs = {EPOCHS}; AdamW lr {LR_HEAD}/{LR_BACKBONE}, wd {WEIGHT_DECAY}, label smoothing {LABEL_SMOOTHING}")
    comp0 = compute_complexity(model.to(DEVICE), od, "HEASNet")
    best_path = os.path.join(od, "best_HEASNet.pt"); hist_path = os.path.join(od, "training_history.csv"); peak = None
    if REUSE_CHECKPOINT and os.path.exists(best_path) and os.path.exists(hist_path):
        print("Existing checkpoint found: skipping training and re-using it."); model.load_state_dict(torch.load(best_path, map_location=DEVICE, weights_only=True)); h = pd.read_csv(hist_path)
        best_ep = int(h.val_accuracy.idxmax() + 1); best_acc = float(h.val_accuracy.max())
    else: model, h, best_ep, best_acc, best_path, peak = train_model(model, trl, val, od, "HEASNet")
    timing = {"total_training_and_validation_seconds": float(h.epoch_seconds.sum()), "total_minutes": float(h.epoch_seconds.sum() / 60), "mean_epoch_seconds": float(h.epoch_seconds.mean()),
              "mean_head_epoch_seconds": float(h.epoch_seconds[:HEAD_WARMUP_EPOCHS].mean()), "mean_finetune_epoch_seconds": float(h.epoch_seconds[HEAD_WARMUP_EPOCHS:].mean()), "peak_gpu_allocated_MB_training": peak,
              "best_epoch": best_ep, "best_val_accuracy": best_acc}
    json.dump(timing, open(os.path.join(od, "training_timing.json"), "w"), indent=2); print("\nTRAINING TIMING"); print(json.dumps(timing, indent=2)); plot_training(h, od, ds, best_ep)
    bench = benchmark_speed(model, od, "HEASNet"); plot_efficiency(comp0, bench, od, ds)
    loaders = [DataLoader(d, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY, persistent_workers=(NUM_WORKERS > 0)) for d in dsets]
    R = evaluate_and_report(model, loaders, dsets, classes, od, ds, tr)
    tag = f"val{100*best_acc:.2f}_test{100*R['test']['metrics']['accuracy']:.2f}"; shutil.copy(best_path, os.path.join(od, f"HEASNet_best_{tag}.pt"))
    banner(f"{ds}: EXPLAINABLE AI", "-"); gradcam_galleries(model, dsets[2], R["test"]["probs"], R["test"]["y"], classes, od)
    corr = attention_analysis(model, dsets[2], R["test"]["probs"], R["test"]["y"], classes, od); faith = faithfulness(model, dsets[2], R["test"]["probs"], R["test"]["y"], classes, od)
    dup = near_duplicate_analysis(ds, tr, va, te, R["test"]["y"], R["test"]["probs"], od)
    CTX[ds] = dict(name=ds, out_dir=od, classes=classes, tr=tr, va=va, te=te, counts=ct, audit=aud, imbalance=imb, hist=h, timing=timing, R=R, complexity=comp0, bench=bench, corr=corr, faith=faith, dup=dup, best_epoch=best_ep, best_val_acc=best_acc, n_corrupt=n_bad)
    summ = {"dataset": ds, "num_images": int(len(df)), "num_classes": len(classes), "classes": classes, "train": len(tr), "validation": len(va), "test": len(te), "best_epoch": best_ep, "best_val_accuracy": best_acc, "timing": timing,
            "split_metrics": {k: R[k]["metrics"] for k in ["train", "validation", "test"]}, "statistics": R["stat"], "calibration": R["cal"], "near_duplicates": dup, "complexity": {k: v for k, v in comp0.items() if k != "breakdown"}}
    json.dump(summ, open(os.path.join(od, "FINAL_EXPERIMENT_SUMMARY.json"), "w"), indent=2, default=str)
    banner(f"FINAL RESULT: {ds}", "="); print(pd.Series(R["test"]["metrics"]).round(5).to_string()); print(f"Best validation accuracy: {best_acc:.4f} at epoch {best_ep}")
    del model; gc.collect(); torch.cuda.empty_cache() if torch.cuda.is_available() else None
    return summ

# ======================================== OPTIONAL EXTRA EXPERIMENTS ========================================
def run_variant(c, name, builder, seed=SEED, opt="adamw", lr_head=LR_HEAD, lr_bb=LR_BACKBONE, epochs=EPOCHS, warmup=HEAD_WARMUP_EPOCHS):
    od = os.path.join(c["out_dir"], "extras"); os.makedirs(od, exist_ok=True); safe = re.sub(r"[^A-Za-z0-9_.+-]", "_", name); jf = os.path.join(od, f"variant_{safe}.json")
    if os.path.exists(jf): print(f"[{name}] already done, loading"); return json.load(open(jf))
    banner(f"EXTRA RUN: {name} | {c['name']} | seed {seed} | {epochs} epochs | opt {opt} | lr {lr_head}", "-"); seed_everything(seed)
    trl, val, tel, _ = make_loaders(c["tr"], c["va"], c["te"], seed); model = builder().to(DEVICE)
    model, h, be, ba, bp, peak = train_model(model, trl, val, od, safe, lr_head, lr_bb, opt, epochs, min(warmup, epochs), verbose=False, history_name=f"history_{safe}.csv")
    pv, yv = predict(model, val); pt, yt = predict(model, tel); mv, _, _ = compute_metrics(yv, pv, c["classes"]); mt, _, _ = compute_metrics(yt, pt, c["classes"])
    comp = compute_complexity(model, od, safe, save=False); x = torch.randn(1, 3, IMG_SIZE, IMG_SIZE, device=DEVICE)
    if DEVICE.type == "cuda": torch.cuda.reset_peak_memory_stats()
    def f():
        with torch.inference_mode(): model(x)
    ts = _timeit(f, 5 if QUICK else 40, 1); pm = float(torch.cuda.max_memory_allocated() / 1e6) if DEVICE.type == "cuda" else None
    row = {"name": name, "dataset": c["name"], "seed": seed, "optimizer": opt, "lr_head": lr_head, "epochs": epochs, "best_epoch": be, "best_val_accuracy": ba, "params_M": comp["parameters_million"], "gmacs": comp["gmacs"],
           "latency_ms_batch1": float(ts.mean()), "peak_gpu_MB_inference": pm, "peak_gpu_MB_training": peak, "train_minutes": float(h.epoch_seconds.sum() / 60),
           **{f"val_{k}": v for k, v in mv.items() if k in ["accuracy", "macro_precision", "macro_recall", "macro_f1"]}, **{f"test_{k}": v for k, v in mt.items()}}
    np.savez_compressed(os.path.join(od, f"test_predictions_{safe}.npz"), probs=pt, y=yt); json.dump(row, open(jf, "w"), indent=2, default=str)
    print(f"[{name}] test acc {mt['accuracy']:.4f} | macro F1 {mt['macro_f1']:.4f} | val acc {ba:.4f} | {row['train_minutes']:.1f} min")
    del model; gc.collect(); torch.cuda.empty_cache() if torch.cuda.is_available() else None
    return row

def _pt(c, name): return np.load(os.path.join(c["out_dir"], "extras", f"test_predictions_{re.sub(r'[^A-Za-z0-9_.+-]', '_', name)}.npz"))["probs"]

def run_extras(c):
    ds = c["name"]; K = len(c["classes"]); od = os.path.join(c["out_dir"], "extras"); os.makedirs(od, exist_ok=True); skipped = []; y = c["R"]["test"]["y"]; main_p = c["R"]["test"]["probs"]
    main_row = {"name": "HEASNet (proposed)", "params_M": c["complexity"]["parameters_million"], "gmacs": c["complexity"]["gmacs"], **{f"test_{k}": v for k, v in c["R"]["test"]["metrics"].items()},
                "val_accuracy": c["best_val_acc"], "latency_ms_batch1": float(c["bench"].query("batch_size==1 and precision=='fp32'").mean_ms_per_image.iloc[0])}
    def guard(what, hrs=0.9):
        if not time_left_ok(hrs): skipped.append(what); print(f"!! time budget: skipping {what}"); return False
        return True
    if RUN_ABLATION:
        rows = []
        for nm, se, wi, ax in [("Base model (EfficientNetV2-S)", 0, 0, 0), ("+ SE", 1, 0, 0), ("+ Window attention", 0, 1, 0), ("+ Window + SE", 1, 1, 0), ("+ Axial", 0, 0, 1), ("+ SE + Axial", 1, 0, 1)]:
            if not guard(f"ablation {nm} {ds}"): continue
            r = run_variant(c, f"abl_{nm}", lambda se=se, wi=wi, ax=ax: HEASNet(K, USE_PRETRAINED, bool(se), bool(wi), bool(ax))); r["name"] = nm; rows.append(r)
        if rows:
            for r in rows: r["McNemar_p_vs_full"] = mcnemar_exact(_pt(c, "abl_" + r["name"]).argmax(1) == y, main_p.argmax(1) == y)["exact_p"]
            d = pd.DataFrame(rows + [main_row]); d.to_csv(os.path.join(od, "ablation_results.csv"), index=False); print("\nABLATION RESULTS"); print(d[["name", "params_M", "gmacs", "test_accuracy", "test_macro_f1", "test_macro_roc_auc_ovr", "best_val_accuracy"]].round(4).to_string(index=False))
            fig, ax = plt.subplots(figsize=(9, 4.8)); a = d.test_accuracy.values * 100; ax.barh(range(len(d)), a, color=[C_TRAIN] * (len(d) - 1) + [C_VAL]); ax.set_yticks(range(len(d))); ax.set_yticklabels(d.name); ax.invert_yaxis()
            ax.set_xlim(max(0, a.min() - 3), 100); ax.set_xlabel("Test accuracy (%)"); [ax.text(v + .1, i, f"{v:.2f}", va="center", fontsize=8) for i, v in enumerate(a)]; ax.set_title(f"{ds}: ablation (single run each)"); fig.tight_layout(); savefig(fig, os.path.join(od, "Fig_26_ablation.png"))
    if RUN_BASELINES:
        rows = []
        for nm in ["ResNet50", "EfficientNetV2-S", "ConvNeXt-T", "ViT-B/16"]:
            if not guard(f"baseline {nm} {ds}", 1.1): continue
            rows.append(run_variant(c, f"base_{nm}", lambda nm=nm: BaselineNet(nm, K, USE_PRETRAINED)) | {"name": nm})
        if rows:
            for r in rows: r["McNemar_p_vs_HEASNet"] = mcnemar_exact(_pt(c, "base_" + r["name"]).argmax(1) == y, main_p.argmax(1) == y)["exact_p"]
            d = pd.DataFrame(rows + [main_row]); d.to_csv(os.path.join(od, "baselines_results.csv"), index=False); print("\nBASELINES"); print(d[["name", "params_M", "gmacs", "latency_ms_batch1", "peak_gpu_MB_inference", "test_accuracy", "test_macro_f1", "test_macro_roc_auc_ovr"]].round(4).to_string(index=False))
            fig, ax = plt.subplots(figsize=(8, 4.6)); a = d.test_accuracy.values * 100; ax.bar(d.name, a, color=[C_TRAIN] * (len(d) - 1) + [C_VAL]); ax.set_ylim(max(0, a.min() - 4), 100.5); ax.set_ylabel("Test accuracy (%)")
            [ax.text(i, v + .1, f"{v:.2f}", ha="center", fontsize=8) for i, v in enumerate(a)]; plt.xticks(rotation=20); ax.set_title(f"{ds}: HEASNet vs retrained baselines (same protocol)"); fig.tight_layout(); savefig(fig, os.path.join(od, "Fig_27_baselines.png"))
    if REPEAT_SEEDS:
        rows = [{**main_row, "seed": SEED}]
        for sd in REPEAT_SEEDS:
            if not guard(f"seed {sd} {ds}"): continue
            rows.append(run_variant(c, f"seed{sd}", lambda: HEASNet(K, USE_PRETRAINED), seed=sd))
        if len(rows) > 1:
            d = pd.DataFrame(rows); d.to_csv(os.path.join(od, "seed_runs.csv"), index=False); ks = ["test_accuracy", "test_macro_f1", "test_macro_roc_auc_ovr", "test_cohen_kappa", "test_mcc", "test_ece_15_bins"]
            s = pd.DataFrame({"mean": d[ks].mean(), "sd": d[ks].std(ddof=1)}); s.to_csv(os.path.join(od, "seed_stability_mean_sd.csv")); print("\nSEED STABILITY (mean, SD)"); print(s.round(5).to_string())
    if RUN_SWEEPS:
        rows = []
        for lr in LR_SWEEP:
            if guard(f"lr sweep {lr} {ds}", .6): rows.append(run_variant(c, f"lr_{lr}", lambda: HEASNet(K, USE_PRETRAINED), lr_head=lr, lr_bb=lr * .1, epochs=SWEEP_EPOCHS))
        if rows: d = pd.DataFrame(rows); d.to_csv(os.path.join(od, "sweep_learning_rate.csv"), index=False); print("\nLEARNING-RATE SWEEP (validation)"); print(d[["name", "lr_head", "val_macro_precision", "val_macro_recall", "val_macro_f1", "val_accuracy"]].round(4).to_string(index=False))
        rows = []
        for o in OPT_SWEEP:
            if guard(f"opt sweep {o} {ds}", .6): rows.append(run_variant(c, f"opt_{o}", lambda: HEASNet(K, USE_PRETRAINED), opt=o, epochs=SWEEP_EPOCHS))
        if rows: d = pd.DataFrame(rows); d.to_csv(os.path.join(od, "sweep_optimizer.csv"), index=False); print("\nOPTIMISER SWEEP (validation; AdamW = the 1e-4 row of the learning-rate sweep)"); print(d[["name", "val_macro_precision", "val_macro_recall", "val_macro_f1", "val_accuracy"]].round(4).to_string(index=False))
    if skipped: open(os.path.join(od, "SKIPPED_DUE_TO_TIME_BUDGET.txt"), "w").write("\n".join(skipped)); print("Skipped due to time:", skipped)

# ======================================== CROSS-DATASET ========================================
def cross_dataset(names):
    banner("CROSS-DATASET COMPARISON", "=")
    od = os.path.join(OUTPUT_ROOT, "CROSS_DATASET"); os.makedirs(od, exist_ok=True)
    keys = ["accuracy", "macro_precision", "macro_recall", "macro_f1", "weighted_f1", "micro_roc_auc", "macro_roc_auc_ovr", "cohen_kappa", "mcc", "log_loss", "brier_score_probability_MSE", "ece_15_bins"]
    rows = []
    for n in names:
        c = CTX[n]; m = c["R"]["test"]["metrics"]; ci = c["R"]["ci"]
        r = {"dataset": n, "classes": len(c["classes"]), "train": len(c["tr"]), "val": len(c["va"]), "test": len(c["te"]), "params_M": c["complexity"]["parameters_million"], "gmacs": c["complexity"]["gmacs"],
             "best_epoch": c["best_epoch"], "best_val_acc": c["best_val_acc"], "train_minutes": c["timing"]["total_minutes"]}
        r.update({k: m.get(k) for k in keys}); r["acc_ci_low"], r["acc_ci_high"] = ci["accuracy"]; rows.append(r)
    d = pd.DataFrame(rows); d.to_csv(os.path.join(od, "cross_dataset_summary.csv"), index=False); print(d.T.to_string(header=False))
    if len(names) > 1:
        fig, ax = plt.subplots(figsize=(11, 4.8)); w = .8 / len(names); sh = ["accuracy", "macro_precision", "macro_recall", "macro_f1", "macro_roc_auc_ovr", "cohen_kappa", "mcc"]
        for i, n in enumerate(names):
            v = [CTX[n]["R"]["test"]["metrics"][k] for k in sh]; b = ax.bar(np.arange(len(sh)) + i * w, v, w, label=n, color=[C_TRAIN, C_VAL, C_TEST][i % 3])
            for rect, t in zip(b, v): ax.text(rect.get_x() + rect.get_width() / 2, t + .002, f"{t:.3f}", ha="center", fontsize=7)
        ax.set_xticks(np.arange(len(sh)) + w * (len(names) - 1) / 2); ax.set_xticklabels(["Accuracy", "Precision", "Recall", "F1", "AUC", "Kappa", "MCC"]); ax.set_ylim(.85, 1.01); ax.legend(); ax.set_title("Test performance across datasets"); fig.tight_layout(); savefig(fig, os.path.join(od, "Fig_28_cross_dataset_metrics.png"))
        fig, axs = plt.subplots(1, 2, figsize=(11, 4)); 
        for n, col in zip(names, [C_TRAIN, C_VAL]):
            h = CTX[n]["hist"]; axs[0].plot(h.epoch, h.val_accuracy * 100, label=n, color=col); axs[1].plot(h.epoch, h.val_loss, label=n, color=col)
        axs[0].set(xlabel="Epoch", ylabel="Validation accuracy (%)", title="Validation accuracy"); axs[1].set(xlabel="Epoch", ylabel="Validation loss", title="Validation loss"); axs[0].legend(); fig.tight_layout(); savefig(fig, os.path.join(od, "Fig_29_cross_dataset_validation_curves.png"))
    if "HyperKvasir" in CTX and "Kvasir-V2" in CTX:
        try:
            hk, kv = CTX["HyperKvasir"], CTX["Kvasir-V2"]; hh = hash_many(pd.concat([hk["tr"], hk["va"], hk["te"]]).path.tolist(), "HyperKvasir all"); hk_ = hash_many(kv["te"].path.tolist(), "Kvasir test")
            dm = min_hamming(hk_, hh); ov = dm <= DUP_HAMMING; ok = kv["R"]["test"]["probs"].argmax(1) == kv["R"]["test"]["y"]
            res = {"kvasir_test_images": int(len(ov)), "overlap_with_hyperkvasir": int(ov.sum()), "overlap_percent": float(100 * ov.mean()), "kvasir_acc_on_overlap": float(ok[ov].mean()) if ov.any() else None, "kvasir_acc_non_overlap": float(ok[~ov].mean()) if (~ov).any() else None}
            json.dump(res, open(os.path.join(od, "kvasir_vs_hyperkvasir_overlap.json"), "w"), indent=2); print("Kvasir-V2 test vs HyperKvasir overlap:", res)
        except Exception as e: print("cross overlap skipped:", e)
    return d

# ======================================== PAPER TABLES / NUMBERS / XLSX ========================================
def write_paper_tables(names):
    L = []; P = L.append
    P("PAPER-READY TABLES (all numbers computed by this run)\n" + "=" * 70)
    for n in names:
        c = CTX[n]; m = c["R"]["test"]["metrics"]; ci = c["R"]["ci"]; P(f"\n### {n}: overall test metrics (95% bootstrap CI, {BOOTSTRAP_REPS} resamples)")
        for k, v in ci.items(): P(f"  {k:<22} {m[k]:.4f}   [{v[0]:.4f}, {v[1]:.4f}]")
        P(f"  precision/recall/F1 macro: {100*m['macro_precision']:.2f} / {100*m['macro_recall']:.2f} / {100*m['macro_f1']:.2f}   weighted F1 {100*m['weighted_f1']:.2f}")
        P(f"  kappa {m['cohen_kappa']:.4f} | MCC {m['mcc']:.4f} | macro AUC {m['macro_roc_auc_ovr']:.4f} | micro AUC {m['micro_roc_auc']:.4f} | log loss {m['log_loss']:.4f} | Brier(MSE) {m['brier_score_probability_MSE']:.4f} | ECE {m['ece_15_bins']:.4f}")
        P(f"\n  class-wise test metrics:\n" + c["R"]["test"]["pcl"].round(4).to_string(index=False))
        P(f"\n  train/val/test accuracy: " + " / ".join(f"{100*c['R'][s]['metrics']['accuracy']:.2f}" for s in ["train", "validation", "test"]))
        P(f"  best validation acc {100*c['best_val_acc']:.2f}% at epoch {c['best_epoch']} | training {c['timing']['total_minutes']:.1f} min | mean epoch {c['timing']['mean_epoch_seconds']:.1f}s")
        P(f"  params {c['complexity']['parameters_million']:.3f} M | GMACs {c['complexity']['gmacs']} | GFLOPs {c['complexity'].get('gflops')}")
        P("  speed/memory:\n" + c["bench"].round(3).to_string(index=False))
        P("  near-duplicates: " + json.dumps(c["dup"]))
        P("  top confusions:\n" + (c["R"]["top_conf"].to_string(index=False) if hasattr(c["R"]["top_conf"], "to_string") else str(c["R"]["top_conf"])))
    open(os.path.join(OUTPUT_ROOT, "PAPER_READY_TABLES.txt"), "w", encoding="utf-8").write("\n".join(L)); print("\n".join(L[:6]), "\n... (full text in PAPER_READY_TABLES.txt)")

def write_numbers_and_xlsx(names):
    nums = {n: json.load(open(os.path.join(CTX[n]["out_dir"], "FINAL_EXPERIMENT_SUMMARY.json"))) for n in names}
    json.dump(nums, open(os.path.join(OUTPUT_ROOT, "manuscript_numbers.json"), "w"), indent=2, default=str)
    try:
        with pd.ExcelWriter(os.path.join(OUTPUT_ROOT, "ALL_RESULTS_TABLES.xlsx")) as xw:
            for n in names:
                tag = n.replace("-", "")[:12]
                for fn in sorted(os.listdir(CTX[n]["out_dir"])):
                    if fn.endswith(".csv") and os.path.getsize(os.path.join(CTX[n]["out_dir"], fn)) < 2_000_000 and not fn.startswith(("exact_image", "near_dup")):
                        try: pd.read_csv(os.path.join(CTX[n]["out_dir"], fn)).to_excel(xw, sheet_name=f"{tag}_{fn[:-4]}"[:31], index=False)
                        except Exception: pass
        print("Excel workbook written.")
    except Exception as e: print("xlsx skipped:", e)

# ======================================== ZIP ========================================
def make_zip(final=False):
    zp = ZIP_BASE + ".zip"
    if os.path.exists(zp): os.remove(zp)
    n = 0
    with zipfile.ZipFile(zp, "w", zipfile.ZIP_DEFLATED, compresslevel=6) as z:
        for r, _, fs in os.walk(OUTPUT_ROOT):
            for f in fs:
                fp = os.path.join(r, f)
                if f.endswith(".pt") and not (INCLUDE_ALL_CHECKPOINTS_IN_ZIP or f == "best_HEASNet.pt"): continue
                if f == "FULL_NOTEBOOK_LOG.txt": _LOG.flush()
                z.write(fp, os.path.relpath(fp, os.path.dirname(OUTPUT_ROOT))); n += 1
    print(f"ZIP written ({'FINAL' if final else 'checkpoint'}): {zp} | {n} files | {os.path.getsize(zp)/1e6:.1f} MB"); return zp

def list_outputs():
    banner("FILES SAVED (relative path, size)", "=")
    tot = 0
    for r, _, fs in sorted(os.walk(OUTPUT_ROOT)):
        for f in sorted(fs):
            s = os.path.getsize(os.path.join(r, f)); tot += s; print(f"{os.path.relpath(os.path.join(r, f), OUTPUT_ROOT):<80} {s/1e3:>10.1f} KB")
    print(f"TOTAL {tot/1e6:.1f} MB")

# ======================================== MAIN ========================================
def main():
    banner(f"HEASNet FINAL ALL-IN-ONE RUN | device {DEVICE} | {datetime.datetime.now():%Y-%m-%d %H:%M}")
    print("torch", torch.__version__, "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none (CPU)")
    done = []
    for ds, cfg in DATASETS.items():
        try: run_dataset(ds, cfg); done.append(ds)
        except Exception as e:
            import traceback; traceback.print_exc(); print(f"!! {ds} FAILED: {e}")
        try: make_zip()
        except Exception as e: print("zip failed:", e)
    if done:
        try: cross_dataset(done)
        except Exception as e: print("cross-dataset failed:", e)
        try: write_paper_tables(done); write_numbers_and_xlsx(done)
        except Exception as e: print("tables failed:", e)
        make_zip()
    if RUN_EXTRAS:
        for ds in EXTRA_DATASETS:
            if ds in CTX:
                try: banner(f"EXTRA EXPERIMENTS: {ds}", "#"); run_extras(CTX[ds])
                except Exception as e:
                    import traceback; traceback.print_exc(); print(f"!! extras {ds} failed: {e}")
                try: make_zip()
                except Exception as e: print("zip failed:", e)
    banner(f"ALL DONE in {elapsed_h():.2f} h", "=")
    list_outputs(); make_zip(final=True)
    try:
        from IPython.display import FileLink, display; display(FileLink(os.path.relpath(ZIP_BASE + ".zip")))
    except Exception: pass
    print("DOWNLOAD:", ZIP_BASE + ".zip")

main()